# OWSM + sEEG (B2T'25) — versión Google Colab con barrido de experimentos

Adaptación del notebook local para Colab. Cambios principales respecto a la versión local:

## Novedades de esta versión (v5)

Tres cambios, cada uno respondiendo a algo que los resultados anteriores dejaron claro.

**1. Frontend `deep`.** El `linear` que se usó en casi todos los experimentos tenía 197 K parámetros y, peor, **descartaba 3 de cada 4 frames** por decimación (`x[:, ::4, :]`). El frontend original de OWSM tiene ~20 M. El nuevo `deep` es:

```
capa por sesión → Linear(512→d) + LayerNorm + GELU     ← mezcla espacial de electrodos
                → [Conv1d stride 2 → n bloques residuales] × 2   ← temporal, sin descartar frames
                → Linear(d→384) + codificación posicional
```

Con `deep_dim=512, deep_bloques=2` son **8,3 M** de parámetros (más 11,8 M de la capa por sesión con 45 sesiones). La mezcla de electrodos es todos-con-todos, no convolucional: el orden de los electrodos es arbitrario, así que convolucionar sobre ese eje (lo que hace `conv2d`, heredado de tratarlo como eje de frecuencias) no tiene sentido físico aquí.

**2. InterCTC.** Pérdidas CTC auxiliares en las capas 2 y 4 del encoder, compartiendo la misma cabeza. Es la receta estándar de ESPnet contra el subajuste en encoders profundos: da señal de gradiente directa a las capas de abajo en vez de que tenga que propagarse desde el final. Se activa con `interctc_weight=0.3`.

**3. Las 45 sesiones.** `SOLO_SESIONES_COMPLETAS` pasa a `False`. Antes se usaban 41 porque 4 sesiones no tienen `data_val.hdf5` y el filtro las descartaba enteras; ahora aportan datos de entrenamiento y simplemente no entran en validación. (De paso, el nombre de `stats_dir` ahora lleva el número real de sesiones, porque con el filtro activado o no `n_sessions=45` producía dos conjuntos de datos distintos con la misma carpeta de estadísticas.)

## Cómo usarlo

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU L4**. Marca *High RAM* solo si vas a usar `n_sessions >= 30` con `PRELOAD_RAM = True`.
2. Comprueba que `DRIVE_DATA` (celda 1) apunta a tu carpeta `hdf5_data_final` en Drive.
3. Ejecuta las celdas 0 a 10 en orden. Reinicia el entorno cuando te lo pida la celda de instalación.
4. Edita la lista `BARRIDO` (celda 11) y lanza la celda 12.

Los resultados se van acumulando en `resultados.csv` en tu Drive: si Colab te desconecta, vuelves a lanzar y se salta los experimentos ya hechos.

## 0. Instalación

Ejecuta esta celda, **reinicia el entorno** (`Entorno de ejecución → Reiniciar sesión`) y continúa desde la celda 1. Si no reinicias, numpy se queda cargado en la versión antigua y aparecen errores de tipos difíciles de diagnosticar.

In [ ]:
!nvidia-smi

# Instalación (~3 min la primera vez).
# Truco para sesiones siguientes: descarga las ruedas a Drive una sola vez con
#   !pip download -q espnet espnet_model_zoo loralib h5py -d /content/drive/MyDrive/TFG/wheels
# y luego instala offline en ~20 s con
#   !pip install -q --no-index --find-links=/content/drive/MyDrive/TFG/wheels espnet espnet_model_zoo loralib h5py
!pip install -q espnet espnet_model_zoo loralib h5py

import torch
print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU: revisa Entorno de ejecucion > Cambiar tipo de entorno de ejecucion")
print("\n>>> REINICIA EL ENTORNO AHORA y sigue desde la celda 1 <<<")

Tue Aug  4 00:15:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   63C    P0             31W /   72W |     478MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Panel de control

Todo lo configurable vive aquí. `DEFAULTS` son los valores base de un experimento; en el barrido (celda 11) solo indicas lo que cambia respecto a estos.

In [ ]:
# ══ RUTAS ══════════════════════════════════════════════════════════
DRIVE_ROOT  = "/content/drive/MyDrive/TFG"                       # <<< tu carpeta en Drive
DRIVE_DATA  = f"{DRIVE_ROOT}/DatosEEG_crudos/hdf5_data_final"    # <<< donde estan los HDF5
DATA_ROOT   = "/content/data/hdf5_data_final"    # cache en el disco local de la VM
OUTPUT_DIR  = "/content/exp"                     # checkpoints en local, NO en Drive
RESULTS_DIR = f"{DRIVE_ROOT}/resultados"         # solo lo que quieres conservar

# ══ MODELO ═════════════════════════════════════════════════════════
FINETUNE_MODEL = "espnet/owsm_v3.1_ebf_base"       # ~101M params
LANGUAGE       = "eng"
LORA_TARGET    = ["w_1", "w_2", "merge_proj", "linear_q", "linear_k", "linear_v", "linear_out"]  # <-- MIRAR ESTO

# ══ DATOS ══════════════════════════════════════════════════════════
PRELOAD_RAM   = False   # False = lectura perezosa del HDF5 (seguro con 45 sesiones)
                        # True  = todo en RAM (~1.6 MB/trial; necesita High RAM si n_sessions>25)
SOLO_SESIONES_COMPLETAS = False  # False = usar las 45 sesiones. Las 4 que no tienen
                                 # data_val.hdf5 aportan datos de entrenamiento y no
                                 # entran en validacion. True = solo las 41 completas.

# ══ EVALUACION ═════════════════════════════════════════════════════
EVAL_N    = 200   # trials de validacion sobre los que calcular CER/WER al final de cada run  <-- MIRAR ESTO
BEAM_SIZE = 5
EVAL_GREEDY_CTC = True   # ademas del beam search, decodificacion CTC greedy (metrica honesta:
                         # no pasa por el decoder, asi que no puede "hacer trampa" con el
                         # prior del ingles preentrenado)

# ══ CONTROL DEL BARRIDO ════════════════════════════════════════════
SKIP_IF_DONE = True    # no repetir experimentos ya presentes en resultados.csv
GUARDAR_CKPT = True    # copiar el mejor .pth a Drive (~470 MB por experimento).
                       # OJO: con False el checkpoint vive solo en /content y MUERE
                       # con la VM. Despues de un run de 10 h te quedarias con el
                       # log y ninguna forma de volver a evaluar el modelo (ni la
                       # seccion 13 del LM, ni ejemplos para la memoria).

# ══ VALORES POR DEFECTO DE UN EXPERIMENTO ══════════════════════════
DEFAULTS = dict(
    nombre        = "base",     # etiqueta legible para las tablas de la memoria
    n_sessions    = 10,         # 45 disponibles
    max_epoch     = 30,
    warmup        = 100,
    val_igual_train = False,    # True = test de sobreajuste (val == train)

    frontend      = "conv2d",   # "conv2d" | "linear" | "conv1d" | "deep"
    deep_dim      = 512,        # (frontend deep) anchura interna
    deep_bloques  = 2,          # (deep) bloques residuales por etapa de submuestreo
    deep_kernel   = 5,          # (deep) tamano del kernel temporal
    deep_dropout  = 0.1,        # (deep) dropout
    subsample     = 4,          # (deep) submuestreo total: 1, 2, 4 u 8.
                                #        Con bins de 20 ms: 4 -> frames de 80 ms,
                                #        2 -> 40 ms (mas resolucion, 2x de coste)
    aug           = False,      # aumento de datos sobre la senal sEEG (solo en train)
    aug_n_tiempo  = 2,          # cuantas bandas temporales se ponen a cero
    aug_max_tiempo = 0.05,      # anchura maxima de cada banda, como fraccion de T
    aug_n_canales = 2,          # cuantas bandas de electrodos se ponen a cero
    aug_max_canales = 40,       # anchura maxima de cada banda, en canales
    aug_ruido     = 0.0,        # ruido blanco gaussiano, desviacion en unidades de la
                                # senal (que ya viene z-scoreada, asi que std~1).
                                # El baseline de B2T'25 usa 1.0: es SU regularizador
                                # principal, no un detalle.
    aug_offset    = 0.0,        # offset constante por canal, sorteado una vez por trial
                                # (B2T'25 usa 0.2). Imita la deriva de linea base de los
                                # electrodos dentro de una misma sesion.
    suavizado     = 0.0,        # sigma del suavizado gaussiano temporal, en bins de 20 ms.
                                # NO es aumento: se aplica igual en train y en validacion.
                                # 2.0 (~40 ms) es lo que usa la literatura de B2T.
    patience      = None,       # None = agotar max_epoch. int = parar tras N epocas sin
                                # mejorar el criterio (cer_ctc), para no quemar horas
                                # en una meseta.
    interctc_weight = 0.0,      # >0 anade perdidas CTC en capas intermedias del
                                # encoder. 0.3 es el valor habitual en ESPnet.
    interctc_layers = (2, 4),   # el encoder tiene 6 bloques: validos 1..5
    objetivo_ctc  = "bpe",      # "bpe"  = tokens BPE de OWSM (50.002 clases; el original)
                                # "char" = caracteres (~32 clases). Cambia SOLO el objetivo
                                #          del CTC; el decoder de atencion sigue en BPE.
    capa_sesion   = False,      # True = transformada afin por sesion + softsign delante del
                                # frontend (es LA caracteristica del baseline de B2T'25: la
                                # senal del mismo fonema cambia de un dia a otro por deriva
                                # de los electrodos, y 45 sesiones abarcan 20 meses)
    norm          = "trial",    # "trial"  = z-score por trial y canal (lo que hacias)
                                # "ninguna"= usar los HDF5 tal cual (ya vienen z-scoreados
                                #            por bloque; los trabajos publicados no renormalizan)
    unfreeze_encoder = True,
    unfreeze_n_layers = None,   # None = encoder entero; int = solo las N primeras capas
    usar_lora     = True,

    batch_type    = "sorted",   # "sorted" (batch_size) | "numel" (batch_bins)
    batch_size    = 16,
    batch_bins    = 6_000_000,
    accum_grad    = 1,

    lr            = 1e-3,
    enc_lr_scale  = 1.0,        # <1.0 = LR menor para el encoder preentrenado (p.ej. 0.01)
    optim         = "adamw",    # "adam" | "adamw". Con weight_decay > 0 hay que usar adamw:
                                # en adam el decay se suma al gradiente y lo reescala el
                                # denominador adaptativo, asi que casi no regulariza.
    weight_decay  = 1e-2,       # 1e-6 (el valor anterior) es basicamente cero. Con adamw,
                                # 1e-2 es el valor estandar y regulariza de forma suave,
                                # sin destrozar la senal como hace el dropout alto.
    grad_clip     = 25.0,       # OJO: con grad_clip=5 y grad_norm~200, el log mostraba
                                # clip=100.000, o sea que el 100% de los pasos se recortaba
                                # y se tiraba 40x de magnitud del gradiente en cada uno.
    ctc_weight    = 0.3,        # ahora SI afecta al loss de entrenamiento
    ctc_weight_decode = None,   # None = mismo que ctc_weight
    seed          = 2024,
    criterio      = "cer_ctc",  # metrica con la que se elige el mejor checkpoint.
                                # "cer_ctc" = la del sEEG. "loss" = la total, que con
                                # ctc_weight bajo es sobre todo perdida de atencion y
                                # toca fondo en 4-5 epocas por el prior del ingles.
    num_workers   = 0,          # 0 = sin fork. Con HDF5 es lo mas seguro y no cuesta
                                # nada: el modelo es compute-bound y los datos estan en
                                # disco local. Sube a 2 solo si ves la GPU esperando.
)

print("Panel cargado.")

Panel cargado.


## 2. Drive y datos

Tus HDF5 ya están en Drive, así que solo hay que montarlo. Lo que **no** conviene es entrenar leyendo directamente de `/content/drive`: Drive limita las operaciones de E/S por fichero y acaba dando `OSError: [Errno 5] Input/output error` a mitad de una época.

La solución es una caché: las sesiones se copian de Drive al disco local de la VM **solo cuando un experimento las necesita**. Si el barrido empieza con 2 sesiones, se copian 2 (unos segundos) y no las 45.

Esta celda solo monta Drive y comprueba qué hay. La copia ocurre sola más adelante.

In [ ]:
import os, time, shutil, glob

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

# Localizar la carpeta de datos aunque este en otro sitio dentro de DRIVE_ROOT
if not os.path.isdir(DRIVE_DATA):
    print(f"No esta en {DRIVE_DATA}; buscando 'hdf5_data_final' bajo {DRIVE_ROOT} ...")
    cands = glob.glob(f"{DRIVE_ROOT}/**/hdf5_data_final", recursive=True)
    if cands:
        DRIVE_DATA = cands[0]
        print("Encontrada:", DRIVE_DATA)
    else:
        raise FileNotFoundError(
            f"No encuentro la carpeta hdf5_data_final dentro de {DRIVE_ROOT}.\n"
            f"Ajusta DRIVE_ROOT / DRIVE_DATA en el panel de control."
        )

# Inventario: que sesiones hay y cuales tienen train Y val
inventario, sin_val, bytes_tot = [], [], 0
for s in sorted(os.listdir(DRIVE_DATA)):
    d = os.path.join(DRIVE_DATA, s)
    if not os.path.isdir(d):
        continue
    tr = os.path.join(d, "data_train.hdf5")
    va = os.path.join(d, "data_val.hdf5")
    if not os.path.exists(tr):
        continue
    inventario.append(s)
    if not os.path.exists(va):
        sin_val.append(s)
    bytes_tot += os.path.getsize(tr) + (os.path.getsize(va) if os.path.exists(va) else 0)

print(f"\n{len(inventario)} sesiones con data_train.hdf5 · {bytes_tot/1e9:.1f} GB (train+val)")
if sin_val:
    print(f"{len(sin_val)} sin data_val.hdf5: {sin_val}")
    print("  -> con SOLO_SESIONES_COMPLETAS=True se ignoran (evita que train y val no cuadren)")
print("\nPrimeras sesiones:", inventario[:5])
print("Disco libre en la VM:")
os.system("df -h /content | tail -1")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

45 sesiones con data_train.hdf5 · 11.2 GB (train+val)
4 sin data_val.hdf5: ['t15.2023.08.11', 't15.2024.03.03', 't15.2024.04.25', 't15.2024.04.28']
  -> con SOLO_SESIONES_COMPLETAS=True se ignoran (evita que train y val no cuadren)

Primeras sesiones: ['t15.2023.08.11', 't15.2023.08.13', 't15.2023.08.18', 't15.2023.08.20', 't15.2023.08.25']
Disco libre en la VM:


0

## 3. Imports, dispositivo y utilidades

In [ ]:
import string, re, glob, argparse, logging, gc, copy, json, math
import h5py
import numpy as np
import pandas as pd
import torch
import loralib

import espnetez as ez
from espnet2.bin.s2t_inference import Speech2Text
from espnet2.layers.create_adapter_fn import create_lora_adapter
from espnet.nets.pytorch_backend.transformer.subsampling import Conv2dSubsampling

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
NGPU    = 1 if torch.cuda.is_available() else 0
USE_AMP = torch.cuda.is_available()

GPU_NAME = torch.cuda.get_device_name(0) if NGPU else "CPU"

# Tarifas aproximadas de compute units/hora (medidas en marzo de 2026; pueden variar)
TARIFAS = {"T4": 1.19, "L4": 1.71, "A100-SXM4-40GB": 5.40, "A100-SXM4-80GB": 7.52,
           "RTX PRO 6000": 8.71, "H100": 9.0}
def units_por_hora(nombre_gpu):
    for k, v in TARIFAS.items():
        if k.lower().replace("-", " ") in nombre_gpu.lower().replace("-", " "):
            return v
    return float("nan")
CU_H = units_por_hora(GPU_NAME)

print(f"Dispositivo: {DEVICE} · {GPU_NAME} · AMP: {USE_AMP}")
print(f"Coste estimado: {CU_H} compute units/hora (~${CU_H*0.10:.2f}/h)")


def normaliza(x, modo="trial"):
    """Normalizacion de la senal sEEG antes de entrar al modelo.

    Los HDF5 de B2T'25 ya vienen binneados a 20 ms y z-scoreados por bloque, asi que
    "ninguna" reproduce lo que hacen los trabajos publicados; "trial" vuelve a normalizar
    por trial y canal (util para comparar, pero borra la amplitud relativa entre trials).
    """
    x = x.astype(np.float32)
    if modo == "ninguna":
        return x
    return (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def decode_transcription(arr):
    arr = np.asarray(arr).ravel()
    return "".join(chr(int(x)) for x in arr if int(x) != 0)

Dispositivo: cuda · NVIDIA L4 · AMP: True
Coste estimado: 1.71 compute units/hora (~$0.17/h)


## 4. Datasets (carga perezosa)

`SeeGDataset` guarda solo un **índice** `(fichero, clave)` y lee cada trial del HDF5 cuando hace falta. Así las 45 sesiones (~17 GB en float32) no tienen que caber en RAM. El handle de h5py se reabre por proceso para que funcione con `num_workers > 0`.

Con `PRELOAD_RAM = True` vuelve al comportamiento local (todo en memoria), que es algo más rápido si tienes RAM de sobra.

In [ ]:
class SeeGDataset(torch.utils.data.Dataset):
    """Lee trials del HDF5 bajo demanda. Si preload=True los cachea todos en RAM."""

    def __init__(self, index, preload=False):
        self.index = index          # [(ruta_hdf5, clave), ...]
        self.preload = preload
        self._files = {}
        self._pid = os.getpid()
        self._cache = None
        if preload:
            self._cache = [self._leer(i) for i in range(len(index))]
            self.cerrar()

    def _handle(self, path):
        if os.getpid() != self._pid:      # tras un fork, los handles del padre no valen
            self._files, self._pid = {}, os.getpid()
        if path not in self._files:
            self._files[path] = h5py.File(path, "r")
        return self._files[path]

    def cerrar(self):
        """Cierra los handles. IMPRESCINDIBLE antes de que el DataLoader haga fork:
        la libreria HDF5 no es fork-safe y heredar handles abiertos puede colgar el proceso."""
        for f in self._files.values():
            try: f.close()
            except Exception: pass
        self._files = {}

    def _leer(self, idx):
        path, key, sid = self.index[idx]
        t = self._handle(path)[key]
        return {"input_features": t["input_features"][:],
                "text_raw": decode_transcription(t["transcription"][()]),
                "session_idx": sid}

    def textos(self):
        """Solo las transcripciones, SIN leer input_features.

        Leer un trial entero cuesta ~1.6 MB; con 10.000 trials serian ~16 GB de
        disco para quedarse con una frase de cada uno. Esto abre cada fichero una
        sola vez y lee unicamente el dataset 'transcription'.
        """
        out = [None] * len(self.index)
        orden = sorted(range(len(self.index)), key=lambda i: self.index[i][0])
        actual, fh = None, None
        try:
            for i in orden:
                path, key, _ = self.index[i]
                if path != actual:
                    if fh is not None:
                        fh.close()
                    fh, actual = h5py.File(path, "r"), path
                out[i] = decode_transcription(fh[key]["transcription"][()])
        finally:
            if fh is not None:
                fh.close()
        return out

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        d = self._cache[idx] if self._cache is not None else self._leer(idx)
        text_lower = d["text_raw"].lower()
        return {
            "input_features": d["input_features"],
            "session_idx": d["session_idx"],
            "text":      f"<{LANGUAGE}><asr><notimestamps> {text_lower}",
            "text_prev": "<na>",
            "text_ctc":  remove_punctuation(text_lower),
            "text_raw":  d["text_raw"],
        }


def construir_indice(sesiones, split):
    """Recorre las claves de cada HDF5 sin leer los datos.

    Cada entrada es (ruta, clave, indice_de_sesion). El indice es la posicion en la
    lista `sesiones`, la misma para train y val, y es lo que consume la capa por sesion.
    """
    index = []
    for sid, s in enumerate(sesiones):
        path = os.path.join(DATA_ROOT, s, f"data_{split}.hdf5")
        if not os.path.exists(path):
            print(f"  [aviso] no encontrado: {path}")
            continue
        with h5py.File(path, "r") as f:
            index.extend((path, k, sid) for k in sorted(f.keys()))
    return index


def sesiones_disponibles():
    """Sesiones utilizables, listadas desde Drive (la fuente de la verdad).

    Con SOLO_SESIONES_COMPLETAS=False se incluyen tambien las sesiones que solo
    tienen data_train.hdf5: aportan datos de entrenamiento y simplemente no
    contribuyen a la validacion.
    """
    out = []
    for s in sorted(os.listdir(DRIVE_DATA)):
        d = os.path.join(DRIVE_DATA, s)
        if not os.path.isdir(d) or not os.path.exists(os.path.join(d, "data_train.hdf5")):
            continue
        if SOLO_SESIONES_COMPLETAS and not os.path.exists(os.path.join(d, "data_val.hdf5")):
            continue
        out.append(s)
    return out


def copiar_sesion(sesion):
    """Trae data_train/data_val de esa sesion al disco local de la VM (solo la primera vez)."""
    src = os.path.join(DRIVE_DATA, sesion)
    dst = os.path.join(DATA_ROOT, sesion)
    os.makedirs(dst, exist_ok=True)
    for split in ("train", "val"):
        f_src = os.path.join(src, f"data_{split}.hdf5")
        f_dst = os.path.join(dst, f"data_{split}.hdf5")
        if os.path.exists(f_src) and not os.path.exists(f_dst):
            shutil.copy(f_src, f_dst)      # data_test.hdf5 no se copia: no tiene etiquetas
    return dst


def elegir_sesiones(n_sessions):
    ses = sesiones_disponibles()[:n_sessions]
    faltan = [s for s in ses
              if not os.path.exists(os.path.join(DATA_ROOT, s, "data_train.hdf5"))]
    if faltan:
        print(f"Copiando {len(faltan)} sesion(es) de Drive al disco local de la VM...")
        t0 = time.time()
        for i, s in enumerate(faltan, 1):
            copiar_sesion(s)
            print(f"  [{i}/{len(faltan)}] {s}")
        print(f"  copiadas en {time.time()-t0:.0f}s")
    return ses


_CACHE_DATOS = {}

def get_datos(cfg):
    """Devuelve (train_raw, val_raw, stats_dir, n_sesiones_reales).

    La clave de cache incluye todo lo que cambia la FORMA del tensor de entrada
    (capa_sesion anade un canal: 512 -> 513), porque los shape files de ESPnet
    dejan de valer si eso cambia.
    """
    n_sessions      = cfg["n_sessions"]
    val_igual_train = cfg["val_igual_train"]
    clave = (n_sessions, val_igual_train, PRELOAD_RAM, cfg["capa_sesion"],
             cfg["objetivo_ctc"], SOLO_SESIONES_COMPLETAS)
    if clave in _CACHE_DATOS:
        return _CACHE_DATOS[clave]

    sesiones = elegir_sesiones(n_sessions)
    print(f"Sesiones ({len(sesiones)}): {sesiones[:3]}{' ...' if len(sesiones) > 3 else ''}")
    idx_train = construir_indice(sesiones, "train")
    idx_val   = idx_train if val_igual_train else construir_indice(sesiones, "val")

    train_raw = SeeGDataset(idx_train, preload=PRELOAD_RAM)
    val_raw   = train_raw if val_igual_train else SeeGDataset(idx_val, preload=PRELOAD_RAM)

    stats_dir = (f"{OUTPUT_DIR}/stats_s{len(sesiones)}"
                 f"{'_vt' if val_igual_train else ''}"
                 f"{'_sess' if cfg['capa_sesion'] else ''}"
                 f"{'_char' if cfg['objetivo_ctc'] == 'char' else ''}")
    print(f"Train: {len(idx_train)} trials · Val: {len(idx_val)} trials")

    _CACHE_DATOS[clave] = (train_raw, val_raw, stats_dir, len(sesiones))
    return _CACHE_DATOS[clave]


# Comprobacion rapida: copia UNA sesion y lee un trial
_ses = sesiones_disponibles()[:1]
assert _ses, "No hay sesiones utilizables; revisa DRIVE_DATA y SOLO_SESIONES_COMPLETAS"
copiar_sesion(_ses[0])
_ds = SeeGDataset(construir_indice(_ses, "train"))
_it = _ds[0]
_x  = _it["input_features"]
print(f"\nSesion de prueba: {_ses[0]} ({len(_ds)} trials)")
print("text    :", _it["text"])
print("text_ctc:", _it["text_ctc"])
print("features:", _x.shape, _x.dtype)
# Los HDF5 de B2T'25 ya vienen z-scoreados por bloque: si esto ya esta cerca de (0, 1),
# volver a normalizar por trial (norm="trial") borra la amplitud relativa entre trials.
print(f"estadisticas del HDF5 crudo: media={_x.mean():+.3f} std={_x.std():.3f} "
      f"min={_x.min():+.2f} max={_x.max():+.2f}")


Sesion de prueba: t15.2023.08.11 (288 trials)
text    : <eng><asr><notimestamps> bring it closer.
text_ctc: bring it closer
features: (321, 512) float32
estadisticas del HDF5 crudo: media=-0.000 std=0.999 min=-2.41 max=+10.00


## 5. Tokenizador y configuración base de OWSM

Se carga el modelo preentrenado **una sola vez** por sesión: de él salen el tokenizador, la config base y una copia de los pesos que reutilizan todos los experimentos.

In [ ]:
# Guarda de orden de ejecucion: esta celda depende de las secciones 3 y 4.
# Si has usado "Ejecutar celda y siguientes" desde aqui, o has saltado alguna
# celda, el error que sale sin esto es un NameError poco informativo.
for _n in ("Speech2Text", "normaliza", "SeeGDataset", "np", "gc"):
    assert _n in globals(), (
        f"Falta '{_n}'. Ejecuta las celdas de las secciones 3 y 4 antes que esta "
        f"(Entorno de ejecucion -> Ejecutar todas).")

pretrained = Speech2Text.from_pretrained(FINETUNE_MODEL, lang_sym=f"<{LANGUAGE}>", beam_size=5)
pretrain_config = vars(pretrained.s2t_train_args)
tokenizer = pretrained.tokenizer
converter = pretrained.converter
_pretrained_state_dict = {k: v.cpu().clone() for k, v in pretrained.s2t_model.state_dict().items()}
del pretrained
gc.collect()

def tokenize(text):
    return np.array(converter.tokens2ids(tokenizer.text2tokens(text)), dtype=np.int64)

BLANK_ID = converter.token_list.index("<blank>") if "<blank>" in converter.token_list else 0

# ── Vocabulario de caracteres para el objetivo CTC alternativo ──────────────
# Solo lo que aparece en las transcripciones de B2T'25 (ingles en minusculas, sin
# puntuacion). El indice 0 se reserva para el blank de CTC.
_CHARS = list("abcdefghijklmnopqrstuvwxyz'") + [" "]
CHAR_BLANK_ID = 0
_CHAR2ID = {c: i + 1 for i, c in enumerate(_CHARS)}      # 0 = blank
_ID2CHAR = {i + 1: c for i, c in enumerate(_CHARS)}
CHAR_VOCAB = len(_CHARS) + 1                              # +1 por el blank

def tokenize_char(text):
    """Trocea en caracteres. Ignora cualquier simbolo fuera del vocabulario."""
    return np.array([_CHAR2ID[c] for c in text.lower() if c in _CHAR2ID], dtype=np.int64)

def destok_char(ids):
    """De ids de caracteres a texto (ya sin blanks; el colapso CTC se hace fuera)."""
    return "".join(_ID2CHAR[i] for i in ids if i in _ID2CHAR)


def objetivo_info(cfg):
    """Devuelve (funcion_tokenizar, id_del_blank, tam_vocab, destok) segun el objetivo."""
    if cfg["objetivo_ctc"] == "char":
        return tokenize_char, CHAR_BLANK_ID, CHAR_VOCAB, destok_char
    return tokenize, BLANK_ID, len(converter.token_list), \
           (lambda ids: tokenizer.tokens2text(converter.ids2tokens(list(ids))))


def enmascarar(x, cfg, rng):
    """Aumento tipo SpecAugment adaptado a sEEG: bandas de tiempo y de canales a cero.

    OJO: se aplica AQUI y no con el specaug de ESPnet a proposito. El specaug de
    ESPnet actua sobre el tensor de entrada completo, que con capa_sesion incluye
    el canal extra con el indice de sesion; enmascararlo lo pondria a 0 y el
    modelo creeria que todos los trials son de la sesion 0. Aqui se enmascaran
    solo los 512 canales de senal, antes de anadir el de sesion.
    """
    # ORDEN IMPORTANTE: primero ruido y offset, despues el enmascarado. Al reves,
    # el ruido rellenaba los canales que se acababan de poner a cero y el
    # enmascarado no enmascaraba nada.
    #
    # Ruido blanco: el regularizador principal del baseline de B2T'25. Con 8.000
    # frases y un encoder de 100 M, obliga a no fiarse de la amplitud exacta.
    if cfg["aug_ruido"] > 0:
        x += rng.normal(0, cfg["aug_ruido"], x.shape).astype(np.float32)
    # Offset constante por canal: un solo valor por electrodo para todo el trial.
    # Imita la deriva de linea base dentro de una sesion, que la capa por sesion
    # no puede corregir porque cambia de un trial a otro.
    if cfg.get("aug_offset", 0) > 0:
        x += rng.normal(0, cfg["aug_offset"], (1, x.shape[1])).astype(np.float32)

    T = x.shape[0]
    for _ in range(cfg["aug_n_tiempo"]):                 # bandas temporales
        w = rng.integers(0, max(1, int(cfg["aug_max_tiempo"] * T)) + 1)
        if w > 0 and T > w:
            t0 = rng.integers(0, T - w)
            x[t0:t0 + w, :] = 0.0
    # Electrodos: subconjunto ALEATORIO, no una banda contigua. El orden de los
    # electrodos es arbitrario (por eso tampoco se convoluciona sobre ese eje),
    # asi que borrar c0:c0+w no tiene el sentido fisico que si tiene en audio,
    # donde la banda de frecuencias contigua es una unidad real.
    n_c = int(cfg["aug_n_canales"] * cfg["aug_max_canales"])
    if n_c > 0:
        x[:, rng.choice(512, size=min(n_c, 512), replace=False)] = 0.0
    return x


def preparar_speech(d, cfg, aug=False, rng=None):
    """Tensor de entrada al modelo: (T, 512) o (T, 513) si hay capa por sesion.

    El indice de sesion viaja como un canal extra constante. Es la forma menos invasiva
    de meterlo: el pipeline de ESPnet solo transporta 'speech', y el padding va al final
    (con ceros), asi que el frame 0 siempre es dato real y de ahi lo lee el frontend.
    """
    x = normaliza(d["input_features"], cfg["norm"])
    if aug:
        x = enmascarar(x, cfg, rng)
    # Suavizado gaussiano temporal. OJO: esto NO es aumento, es preprocesado, asi
    # que va tambien en validacion e inferencia. Va DESPUES del ruido a proposito:
    # asi el suavizado tiene algo que suavizar, que es como lo aplica B2T'25.
    if cfg.get("suavizado", 0) > 0:
        from scipy.ndimage import gaussian_filter1d
        x = gaussian_filter1d(x, cfg["suavizado"], axis=0, mode="nearest").astype(np.float32)
    if cfg["capa_sesion"]:
        sid = np.full((x.shape[0], 1), float(d["session_idx"]), dtype=np.float32)
        x = np.concatenate([x, sid], axis=1)
    return x


def make_data_info(cfg, aug=False):
    # text y text_prev SIEMPRE en BPE: alimentan el decoder de atencion de OWSM.
    # text_ctc usa el tokenizador del objetivo elegido (BPE o caracteres).
    tok_ctc, _, _, _ = objetivo_info(cfg)
    aug = aug and cfg["aug"]
    rng = np.random.default_rng(cfg["seed"]) if aug else None
    return {
        "speech":    lambda d: preparar_speech(d, cfg, aug, rng),
        "text":      lambda d: tokenize(d["text"]),
        "text_prev": lambda d: tokenize(d["text_prev"]),
        "text_ctc":  lambda d: tok_ctc(d["text_ctc"]),
    }

print("Tokenizador y config base cargados")
print("  ctc_weight original de OWSM:", pretrain_config.get("model_conf", {}).get("ctc_weight"))
print(f"  BPE: {len(converter.token_list)} clases (blank id {BLANK_ID}) · "
      f"caracteres: {CHAR_VOCAB} clases (blank id {CHAR_BLANK_ID})")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

INFO:root:config file: /usr/local/lib/python3.12/dist-packages/espnet_model_zoo/models--espnet--owsm_v3.1_ebf_base/snapshots/a7c58ca6851611e4c5b1bd08690f618e7169436c/exp/s2t_train_s2t_ebf_conv2d_size384_e6_d6_piecewise_lr1e-3_warmup60k_flashattn_lessreg_raw_bpe50000/config.yaml
INFO:root:Vocabulary size: 50002
INFO:root:Gradient checkpoint layers: []
INFO:root:Gradient checkpoint layers: []
INFO:root:BatchBeamSearch implementation is selected.
INFO:root:Beam_search: BatchBeamSearch(
  (nn_dict): ModuleDict(
    (decoder): TransformerDecoder(
      (embed): Sequential(
        (0): Embedding(50002, 384)
        (1): PositionalEncoding(
          (dropout): Dropout(p=0.05, inplace=False)
        )
      )
      (after_norm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (output_layer): Linear(in_features=384, out_features=50002, bias=True)
      (decoders): MultiSequential(
        (0): DecoderLayer(
          (self_attn): MultiHeadedAttention(
            (linear_q): Linea

Tokenizador y config base cargados
  ctc_weight original de OWSM: 0.3
  BPE: 50002 clases (blank id 0) · caracteres: 29 clases (blank id 0)


## 6. Construcción del modelo

Tres arreglos respecto a la versión local:

1. **`ctc_weight` se inyecta en `model_conf`.** Antes el modelo se construía desde `pretrain_config`, así que el peso del CTC en el loss era siempre el 0.3 de OWSM, dijeras lo que dijeras en el panel. (Se veía en los logs: `0.3·loss_ctc + 0.7·loss_att = loss`.)
2. **`usar_lora` es opcional**, para poder comparar con/sin adaptadores.
3. **`enc_lr_scale`** parchea `build_optimizers` para poner el encoder preentrenado en un grupo de parámetros con LR más bajo. Con Adam no vale escalar gradientes (es invariante a escala), hace falta grupos de verdad.

**Nota sobre los frontends `linear`/`conv1d`:** heredan de `Conv2dSubsampling` aunque no usen su `__init__`. Es necesario porque el encoder de ESPnet decide si pasarle la máscara de padding al frontend mirando `isinstance(self.embed, Conv2dSubsampling)`; si la clase no hereda de ahí, ESPnet asume que es un embedding que no cambia la longitud de la secuencia (como un `nn.Embedding` de texto) y lo llama sin máscara, lo que revienta con `TypeError: forward() missing 1 required positional argument: 'x_mask'`.

In [ ]:
def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def _subsample_mask(x_mask, subsample, target_len):
    """Recorta/rellena la mascara para que case EXACTAMENTE con la longitud de x tras el submuestreo."""
    if x_mask is None:
        return None
    m = x_mask[:, :, ::subsample]
    if m.size(2) > target_len:
        m = m[:, :, :target_len]
    elif m.size(2) < target_len:
        m = torch.nn.functional.pad(m, (0, target_len - m.size(2)), value=False)
    return m


class CapaSesion(torch.nn.Module):
    """Transformada afin por sesion + softsign (capa 'day-specific' del baseline de B2T'25).

    Por que hace falta: los 45 dias de grabacion abarcan 20 meses y la senal que produce
    un mismo fonema cambia de un dia a otro (deriva de los electrodos, micromovimientos
    del array). Un unico frontend compartido tiene que aprender N mapeos senal->fonema
    contradictorios a la vez y ademas adivinar de que dia es cada trial. Con una matriz
    por sesion, cada dia se alinea a un espacio comun y el resto del modelo ve un unico
    problema coherente.

    Se inicializa a la identidad, asi que al empezar solo aplica el softsign.
    Coste: n_sesiones x 512 x 512 parametros (~2.6 M con 10 sesiones, ~12 M con 45).
    """

    def __init__(self, n_sesiones, dim=512):
        super().__init__()
        self.n_sesiones = n_sesiones
        self.W = torch.nn.Parameter(torch.eye(dim).unsqueeze(0).repeat(n_sesiones, 1, 1))
        self.b = torch.nn.Parameter(torch.zeros(n_sesiones, dim))

    def forward(self, x):
        # x: (B, T, dim+1). El ultimo canal es el indice de sesion, constante en el tiempo.
        sid = x[:, 0, -1].round().long().clamp_(0, self.n_sesiones - 1)
        x = x[:, :, :-1]
        # baddbmm: b[sid] + x @ W[sid], por muestra del batch
        x = torch.baddbmm(self.b[sid].unsqueeze(1), x, self.W[sid])
        return torch.nn.functional.softsign(x)


def build_frontend(kind, pos_enc, capa_sesion=None, cfg=None):
    """Frontend (encoder.embed) que reemplaza al Conv2dSubsampling original de mel.

    IMPORTANTE: heredan de Conv2dSubsampling (aunque no usen su __init__) porque el
    encoder de ESPnet decide si pasar la mascara al embed mirando
    `isinstance(self.embed, Conv2dSubsampling)`. Si no se hereda de ahi, ESPnet llama
    a embed(x) SIN mascara (asumiendo que es un embedding de texto que no submuestrea)
    y revienta con "forward() missing 1 required positional argument: 'x_mask'".
    """
    import torch.nn as nn

    if kind == "conv2d":
        class Conv2dFrontend(Conv2dSubsampling):
            # OJO: ~66 GFLOP/muestra, el 85-90% del coste del modelo.
            def __init__(self):
                super().__init__(512, 384, dropout_rate=0.0, pos_enc=pos_enc)
                self.sesion = capa_sesion
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                return super().forward(x, x_mask)
        return Conv2dFrontend()

    elif kind == "linear":
        class LinearFrontend(Conv2dSubsampling):
            def __init__(self, in_dim=512, out_dim=384, subsample=4):
                torch.nn.Module.__init__(self)   # NO llamar a Conv2dSubsampling.__init__
                self.sesion = capa_sesion
                self.proj = nn.Linear(in_dim, out_dim)
                self.subsample = subsample
                self.pos_enc = pos_enc
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.proj(x)
                x = x[:, ::self.subsample, :]
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, self.subsample, x.size(1))
                return x, x_mask
        return LinearFrontend()

    elif kind == "conv1d":
        class Conv1dFrontend(Conv2dSubsampling):
            def __init__(self, in_dim=512, out_dim=384):
                torch.nn.Module.__init__(self)   # NO llamar a Conv2dSubsampling.__init__
                self.sesion = capa_sesion
                self.conv = nn.Sequential(
                    nn.Conv1d(in_dim, out_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
                    nn.Conv1d(out_dim, out_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
                )
                self.pos_enc = pos_enc
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.conv(x.transpose(1, 2)).transpose(1, 2)
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, 4, x.size(1))
                return x, x_mask
        return Conv1dFrontend()

    elif kind == "deep":
        # Frontend convolucional profundo, pensado para sEEG (no para audio).
        #   1. proyeccion ESPACIAL: Linear(512 -> d) que mezcla los 256 electrodos.
        #      A diferencia de conv2d, que convoluciona a lo largo del eje de
        #      electrodos como si fuera un eje de frecuencias, aqui la mezcla es
        #      todos-con-todos, que es lo correcto cuando el orden de los
        #      electrodos es arbitrario.
        #   2. bloques TEMPORALES Conv1d con LayerNorm + GELU y conexiones
        #      residuales. Las convoluciones ven todos los frames (nada de
        #      decimacion), y el submuestreo se hace con stride.
        #   3. proyeccion final a 384 + codificacion posicional de OWSM.
        d        = (cfg or {}).get("deep_dim", 512)
        n_res    = (cfg or {}).get("deep_bloques", 2)   # bloques residuales por etapa
        kernel   = (cfg or {}).get("deep_kernel", 5)
        dropout  = (cfg or {}).get("deep_dropout", 0.1)
        subsample = (cfg or {}).get("subsample", 4)     # 4 = dos etapas de stride 2

        class BloqueTemporal(nn.Module):
            def __init__(self, dim, k, stride):
                super().__init__()
                self.conv = nn.Conv1d(dim, dim, k, stride=stride, padding=k // 2)
                self.norm = nn.LayerNorm(dim)
                self.act = nn.GELU()
                self.drop = nn.Dropout(dropout)
                self.residual = (stride == 1)
            def forward(self, x):                        # x: (B, T, dim)
                y = self.conv(x.transpose(1, 2)).transpose(1, 2)
                y = self.drop(self.act(self.norm(y)))
                return x + y if self.residual else y

        class FrontendProfundo(Conv2dSubsampling):
            def __init__(self):
                torch.nn.Module.__init__(self)
                self.sesion = capa_sesion
                self.subsample = subsample
                self.espacial = nn.Sequential(
                    nn.Linear(512, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout))
                bloques = []
                n_etapas = {1: 0, 2: 1, 4: 2, 8: 3}[subsample]
                for _ in range(n_etapas):
                    bloques.append(BloqueTemporal(d, kernel, stride=2))
                    bloques += [BloqueTemporal(d, kernel, 1) for _ in range(n_res)]
                if n_etapas == 0:
                    bloques += [BloqueTemporal(d, kernel, 1) for _ in range(n_res)]
                self.bloques = nn.ModuleList(bloques)
                self.salida = nn.Linear(d, 384)
                self.pos_enc = pos_enc

            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.espacial(x)
                # Anular el padding: tras la capa espacial (con sesgo) los frames
                # de relleno dejan de ser cero y se colarian en las convoluciones.
                if x_mask is not None:
                    x = x * x_mask.transpose(1, 2).to(x.dtype)
                for b in self.bloques:
                    x = b(x)
                x = self.salida(x)
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, self.subsample, x.size(1))
                return x, x_mask

        return FrontendProfundo()

    raise ValueError(f"Frontend desconocido: {kind}")


def make_build_model_fn(cfg, n_sesiones=1, verbose=True):
    """Devuelve el build_model_fn que ESPnet-EZ usara para ESTE experimento."""
    def build_model_fn(args):
        from espnet2.tasks.s2t import S2TTask

        # ── ARREGLO: el ctc_weight del panel tiene que llegar al modelo ──
        conf = dict(pretrain_config)
        conf["model_conf"] = {**conf.get("model_conf", {}), "ctc_weight": cfg["ctc_weight"]}

        # ── InterCTC: perdidas CTC auxiliares en capas intermedias del encoder ──
        # Da senal de gradiente directa a las capas de abajo en vez de que tenga
        # que propagarse desde el final. Es la receta estandar contra el
        # subajuste en encoders profundos entrenados con CTC.
        if cfg["interctc_weight"] > 0:
            conf["encoder_conf"] = {**conf.get("encoder_conf", {}),
                                    "interctc_layer_idx": list(cfg["interctc_layers"])}
            conf["model_conf"]["interctc_weight"] = cfg["interctc_weight"]

        model = S2TTask.build_model(argparse.Namespace(**conf))
        model.load_state_dict(_pretrained_state_dict, strict=False)

        # sEEG entra directa: fuera frontend mel y normalizacion global
        model.frontend = None
        model.normalize = None

        pos_enc = model.encoder.embed.out[1]
        sesion = CapaSesion(n_sesiones) if cfg["capa_sesion"] else None
        model.encoder.embed = build_frontend(cfg["frontend"], pos_enc, sesion, cfg)

        # ── objetivo CTC de caracteres: nueva cabeza + error_calculator coherente ──
        if cfg["objetivo_ctc"] == "char" and model.ctc is not None:
            from espnet2.asr.ctc import CTC
            from espnet.nets.pytorch_backend.transformer.subsampling import Conv2dSubsampling  # noqa
            from espnet.nets.e2e_asr_common import ErrorCalculator
            enc_dim = model.ctc.ctc_lo.in_features
            # ctc_lo pasa de (enc_dim -> 50002) a (enc_dim -> ~32). Se entrena desde cero.
            model.ctc = CTC(odim=CHAR_VOCAB, encoder_output_size=enc_dim, zero_infinity=True)
            # El cer_ctc del train.log lo calcula el error_calculator con SU token_list;
            # hay que darle el de caracteres o las cifras del log no querran decir nada.
            char_list = ["<blank>"] + _CHARS
            model.error_calculator = ErrorCalculator(
                char_list, " ", "<blank>", report_cer=True, report_wer=True)
            model.token_list = char_list

        model.train()
        freeze_all(model)
        # LoRA solo tiene sentido con la base CONGELADA: aprende un delta de rango bajo
        # sobre pesos fijos. Si el encoder tambien se entrena, W y BA se solapan y solo
        # se anaden parametros y ciclos de merge/unmerge sin ganar nada.
        if cfg["usar_lora"] and cfg["unfreeze_encoder"] and cfg["unfreeze_n_layers"] is None:
            if verbose:
                print("  [nota] usar_lora=True con el encoder entero descongelado no aporta: "
                      "se desactiva LoRA")
        elif cfg["usar_lora"]:
            create_lora_adapter(model, target_modules=LORA_TARGET)

        for p in model.encoder.embed.parameters():   # frontend nuevo (incl. capa de sesion)
            p.requires_grad = True
        if model.ctc is not None:                    # ESPnet lo pone a None si ctc_weight==0
            for p in model.ctc.parameters():
                p.requires_grad = True

        if cfg["unfreeze_encoder"]:
            if cfg["unfreeze_n_layers"] is None:
                for p in model.encoder.parameters():
                    p.requires_grad = True
            else:
                for layer in model.encoder.encoders[:cfg["unfreeze_n_layers"]]:
                    for p in layer.parameters():
                        p.requires_grad = True

        if verbose:
            total, trainable = count_trainable(model)
            extra = f" + capa_sesion({n_sesiones})" if cfg["capa_sesion"] else ""
            obj = (f"char({CHAR_VOCAB} clases)" if cfg["objetivo_ctc"] == "char"
                   else f"bpe({len(converter.token_list)} clases)")
            print(f"  frontend={cfg['frontend']}{extra} · objetivo_ctc={obj} · "
                  f"lora={cfg['usar_lora']} · "
                  f"encoder={'descongelado' if cfg['unfreeze_encoder'] else 'congelado'} · "
                  f"ctc_weight={cfg['ctc_weight']} · norm={cfg['norm']}")
            print(f"  {trainable:,} entrenables / {total:,} totales ({trainable/total*100:.2f}%)")
            if model.ctc is None:
                print("  [nota] ctc_weight=0 -> ESPnet desactiva la cabeza CTC")
            if getattr(model, "decoder", None) is None:
                print("  [nota] ctc_weight=1 -> ESPnet desactiva el decoder "
                      "(solo CTC; la evaluacion sera greedy)")
        return model

    return build_model_fn


# ── LR diferencial: grupos de parametros en el optimizador ────────────
def set_encoder_lr_scale(scale):
    """scale<1.0 -> el encoder preentrenado entrena con lr*scale; el frontend nuevo, CTC y LoRA con lr."""
    from espnet2.tasks.s2t import S2TTask

    if scale is None or scale == 1.0:
        if "build_optimizers" in S2TTask.__dict__:
            del S2TTask.build_optimizers      # restaura la implementacion original
        return

    OPTIMS = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW, "sgd": torch.optim.SGD}

    def build_optimizers(cls, args, model):
        if args.optim not in OPTIMS:
            raise ValueError(f"enc_lr_scale no soportado con optim={args.optim}")
        base_lr = args.optim_conf.get("lr", 1e-3)
        conf = {k: v for k, v in args.optim_conf.items() if k != "lr"}
        nuevos, preentrenados = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            es_nuevo = (name.startswith("encoder.embed") or name.startswith("ctc.")
                        or "lora_" in name)
            (nuevos if es_nuevo else preentrenados).append(p)
        grupos = [{"params": nuevos, "lr": base_lr},
                  {"params": preentrenados, "lr": base_lr * scale}]
        print(f"  LR diferencial: {len(nuevos)} tensores a {base_lr:g} · "
              f"{len(preentrenados)} a {base_lr*scale:g}")
        return [OPTIMS[args.optim](grupos, lr=base_lr, **conf)]

    S2TTask.build_optimizers = classmethod(build_optimizers)

print("Constructores de modelo listos.")

Constructores de modelo listos.


## 7. Configuración de entrenamiento

Además de lo que ya inyectabas en local, aquí van los ajustes específicos de GPU: `use_amp`, `cudnn_benchmark`, `cudnn_deterministic=False` (el valor por defecto `True` cuesta un 20-30 %), `keep_nbest_models=1` para no llenar el disco y `resume=True` para sobrevivir a las desconexiones de Colab.

In [ ]:
YAML_BASE = """
use_lora: true

rir_scp: null
noise_scp: null
speech_volume_normalize: null
non_linguistic_symbols: null

preprocessor_conf:
  speech_name: speech
  text_name: text

seed: 2024
num_workers: 0
ngpu: 1
batch_type: sorted
batch_size: 8
accum_grad: 1
max_epoch: 1
patience: null
init: null
best_model_criterion:
- [valid, loss, min]
keep_nbest_models: 1
use_amp: true

optim: adam
optim_conf:
    lr: 0.001
    weight_decay: 0.000001
scheduler: warmuplr
scheduler_conf:
    warmup_steps: 20

specaug: null
ctc_weight: 0.0
grad_clip: 5.0
"""

with open("/content/finetune_b2t25.yaml", "w") as f:
    f.write(YAML_BASE)


def build_finetune_config(cfg, stats_dir):
    ft = ez.config.update_finetune_config(
        "s2t", copy.deepcopy(pretrain_config), "/content/finetune_b2t25.yaml")

    # ── entorno ──
    ft["ngpu"] = NGPU
    ft["use_amp"] = USE_AMP
    ft["num_workers"] = cfg["num_workers"]
    # cudnn_benchmark=True es CONTRAPRODUCENTE aqui: con batch_type sorted cada batch
    # tiene una longitud distinta, y cudnn vuelve a buscar el mejor algoritmo para cada
    # forma nueva (143 busquedas exhaustivas en la primera epoca).
    ft["cudnn_benchmark"] = False
    ft["cudnn_deterministic"] = False
    ft["multiple_iterator"] = False
    ft["iterator_type"] = "sequence"
    ft["log_interval"] = 10
    ft["num_iters_per_epoch"] = None
    ft["seed"] = cfg["seed"]
    ft["resume"] = True
    ft["keep_nbest_models"] = 1
    # OJO: en ESPnet, patience NO se mide sobre best_model_criterion. Usa un
    # parametro aparte, early_stopping_criterion, cuyo defecto es
    # ("valid", "loss", "min"). Con sobreajuste el loss de validacion sube
    # mientras el CER sigue bajando, asi que el defecto corta el entrenamiento
    # antes de tiempo: v3 paro en la epoca 38 de 60 con el mejor CER en la 37 y
    # una pendiente de -0.0035 por epoca, o sea aun mejorando.
    #
    # patience_start_epoch evita que se cuente durante el warmup ruidoso (epocas
    # 1-11 en v3 tuvieron delta erratica). Esperar hasta la 15 es seguro.
    ft["patience"] = cfg.get("patience")
    ft["patience_start_epoch"] = 15
    ft["early_stopping_criterion"] = ["valid", "cer_ctc", "min"]

    # ── criterio para elegir el mejor checkpoint ──
    # ARREGLO: con ctc_weight bajo, 'valid loss' es sobre todo perdida de atencion y
    # toca fondo hacia la epoca 4, cuando el cer_ctc todavia esta bajando. Seleccionar
    # por cer_ctc guarda el modelo que de verdad es mejor en la tarea.
    if cfg["criterio"] == "cer_ctc" and cfg["ctc_weight"] > 0:
        ft["best_model_criterion"] = [["valid", "cer_ctc", "min"]]
    else:
        ft["best_model_criterion"] = [["valid", "loss", "min"]]

    # ── batching ──
    ft["batch_type"] = cfg["batch_type"]
    if cfg["batch_type"] == "numel":
        ft["batch_bins"] = cfg["batch_bins"]
    else:
        ft["batch_size"] = cfg["batch_size"]
    ft["accum_grad"] = cfg["accum_grad"]

    # ── optimizacion ──
    ft["max_epoch"] = cfg["max_epoch"]
    ft["scheduler_conf"]["warmup_steps"] = cfg["warmup"]
    ft["optim"] = cfg["optim"]
    ft["optim_conf"]["lr"] = cfg["lr"]
    ft["optim_conf"]["weight_decay"] = cfg["weight_decay"]
    ft["grad_clip"] = cfg["grad_clip"]
    ft["ctc_weight"] = cfg["ctc_weight"]          # informativo, el que manda es model_conf
    ft["model_conf"] = {**ft.get("model_conf", {}), "ctc_weight": cfg["ctc_weight"]}

    # ── shape files ──
    nombres = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
    ft["train_shape_file"] = [f"{stats_dir}/train/{n}" for n in nombres]
    ft["valid_shape_file"] = [f"{stats_dir}/valid/{n}" for n in nombres]
    return ft


def shape_files(stats_dir):
    nombres = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
    return ([f"{stats_dir}/train/{n}" for n in nombres]
            + [f"{stats_dir}/valid/{n}" for n in nombres])

print("Constructor de config listo.")

Constructor de config listo.


## 8. Logs y métricas por época

ESPnet no escribe `train.log` por sí solo cuando se lanza desde un notebook (en las recetas sale de una redirección de shell). Aquí se captura el logger raíz a un fichero por experimento y solo se muestran en pantalla las líneas de resumen de época, así el notebook no se llena de miles de líneas.

`parse_train_log` convierte ese log en un CSV con una fila por época: justo lo que necesitas para las curvas de la memoria.

In [ ]:
class CapturaLog:
    """Redirige el logging de ESPnet a un fichero; en pantalla, solo el resumen por epoca."""

    CLAVES = ("batch:", "epoch results", "epoch started", "Saving", "best",
              "There are no improvements", "Stop training", "The training was finished")

    def __init__(self, path):
        self.path = path
        self.previos = None

    def __enter__(self):
        root = logging.getLogger()
        self.previos = (root.handlers[:], root.level)
        root.handlers = []
        root.setLevel(logging.INFO)

        fh = logging.FileHandler(self.path, mode="a", encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
        root.addHandler(fh)

        claves = self.CLAVES
        class Resumen(logging.Filter):
            def filter(self, record):
                return any(c in record.getMessage() for c in claves)
        sh = logging.StreamHandler()
        sh.addFilter(Resumen())
        sh.setFormatter(logging.Formatter("    %(message)s"))
        root.addHandler(sh)
        return self

    def __exit__(self, *exc):
        root = logging.getLogger()
        for h in root.handlers:
            try: h.close()
            except Exception: pass
        root.handlers, root.level = self.previos
        return False


PAR = re.compile(r"([a-zA-Z_][a-zA-Z_0-9]*)=(-?[\d.]+(?:[eE][-+]?\d+)?)")

def parse_train_log(path):
    """Extrae una fila por epoca del train.log de ESPnet."""
    filas = []
    if not os.path.exists(path):
        return pd.DataFrame()
    with open(path, encoding="utf-8", errors="ignore") as f:
        for linea in f:
            m = re.search(r"(\d+)epoch results:(.*)", linea)
            if not m:
                continue
            epoca, resto = int(m.group(1)), m.group(2)
            fila = {"epoch": epoca}
            partes = re.split(r"\[valid\]", resto)
            for prefijo, trozo in zip(["train_", "valid_"], partes):
                trozo = trozo.replace("[train]", "")
                for k, v in PAR.findall(trozo):
                    if k in ("time", "total_count"):
                        continue
                    fila[prefijo + k] = float(v)
            filas.append(fila)
    df = pd.DataFrame(filas)
    if len(df):
        df = df.drop_duplicates(subset="epoch", keep="last").sort_values("epoch")
    return df

print("Utilidades de log listas.")

Utilidades de log listas.


## 9. Inferencia y CER/WER

Igual que en local (beam search de OWSM reconectado al modelo fine-tuneado), pero el objeto `Speech2Text` se construye **una sola vez** y se reutiliza en todos los experimentos, y los pesos de decodificación (`ctc_weight`) se ajustan sobre la marcha.

In [ ]:
_S2T = None

def get_s2t():
    global _S2T
    if _S2T is None:
        _S2T = Speech2Text.from_pretrained(
            FINETUNE_MODEL, lang_sym=f"<{LANGUAGE}>", task_sym="<asr>",
            beam_size=BEAM_SIZE, ctc_weight=0.3, device=DEVICE, nbest=1)
    return _S2T


def rebind_beam_search(s2t, model, ctc_weight):
    """Repunta decoder y CTC del beam search hacia el modelo fine-tuneado."""
    s2t.s2t_model = model
    bs = s2t.beam_search
    for d in (getattr(bs, "scorers", {}), getattr(bs, "full_scorers", {}),
              getattr(bs, "part_scorers", {})):
        if "decoder" in d:
            d["decoder"] = model.decoder
        if "ctc" in d and hasattr(d["ctc"], "ctc"):
            d["ctc"].ctc = model.ctc
    if hasattr(bs, "nn_dict") and "decoder" in bs.nn_dict:
        bs.nn_dict["decoder"] = model.decoder
    if hasattr(bs, "weights") and ctc_weight is not None:
        bs.weights["ctc"] = ctc_weight
        bs.weights["decoder"] = 1.0 - ctc_weight
    bs.to(device=DEVICE).eval()


def _norm(t):
    t = t.lower().replace("\x00", "")
    t = re.sub(r"<[^>]+>", " ", t)
    t = t.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", t).strip()

def _lev(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            old = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
            prev = old
    return dp[-1]

def cer(ref, hyp):
    r, h = _norm(ref), _norm(hyp)
    return 0.0 if len(r) == 0 else _lev(list(r), list(h)) / len(r)

def wer(ref, hyp):
    r, h = _norm(ref).split(), _norm(hyp).split()
    return 0.0 if len(r) == 0 else _lev(r, h) / len(r)


def find_best_checkpoint(exp_dir):
    for name in ["valid.cer_ctc.best.pth", "valid.loss.best.pth",
                 "valid.acc.best.pth", "train.loss.best.pth"]:
        p = os.path.join(exp_dir, name)
        if os.path.exists(p):
            return p
    cands = sorted(glob.glob(os.path.join(exp_dir, "*epoch.pth")), key=os.path.getmtime)
    if not cands:
        raise FileNotFoundError(f"No hay checkpoints en {exp_dir}")
    return cands[-1]


def cargar_modelo_entrenado(cfg, exp_dir, n_sesiones):
    ckpt = find_best_checkpoint(exp_dir)
    modelo = make_build_model_fn(cfg, n_sesiones, verbose=False)(None).to(DEVICE)
    try:
        state = torch.load(ckpt, map_location=DEVICE, weights_only=True)
    except Exception:
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    faltan = modelo.load_state_dict(state, strict=False)
    if faltan.missing_keys:
        print(f"    [aviso] {len(faltan.missing_keys)} tensores sin cargar del checkpoint, "
              f"p.ej. {faltan.missing_keys[0]}")
    return modelo.eval()


@torch.no_grad()
def evaluar_ctc_greedy(cfg, modelo, val_raw, n=EVAL_N, mostrar=3):
    """Decodificacion CTC greedy: argmax por frame, colapsar repeticiones, quitar blanks.
    No pasa por el decoder, asi que mide SOLO lo que el encoder extrae del sEEG.

    Devuelve DOS cifras, porque no son la misma:
      - cer_ctc_micro: exactamente como lo calcula ESPnet en el train.log (cadena de
        tokens BPE concatenados, sum(errores)/sum(longitudes)). Es la unica comparable
        con la columna cer_ctc de las curvas.
      - cer_texto: sobre el texto detokenizado y normalizado, media por trial. Es la
        que se parece a un CER "de verdad", pero la hunden los trials cortos que fallan.
    """
    from itertools import groupby
    tok_ctc, blank, _, destok = objetivo_info(cfg)
    n = min(n, len(val_raw))
    idxs = np.linspace(0, len(val_raw) - 1, n).astype(int)   # repartidos, no los n primeros

    err_bpe = ref_bpe = 0
    cers_txt, wers_txt, ejemplos = [], [], []
    for i in idxs:
        d = val_raw[int(i)]
        x = preparar_speech(d, cfg)
        speech = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        lens = torch.tensor([speech.shape[1]], dtype=torch.long, device=DEVICE)
        enc, _ = modelo.encode(speech, lens)
        if isinstance(enc, tuple):
            enc = enc[0]
        ids = modelo.ctc.ctc_lo(enc).argmax(-1)[0].tolist()
        hyp_ids = [k for k, _ in groupby(ids) if k != blank]   # colapso CTC estandar
        ref_ids = tok_ctc(d["text_ctc"]).tolist()

        # cer_ctc(micro): identico a como lo calcula ESPnet, sobre la unidad del objetivo
        h = destok(hyp_ids)
        r = destok(ref_ids)
        err_bpe += _lev(list(h), list(r)); ref_bpe += len(r)

        hip = destok(hyp_ids)
        cers_txt.append(cer(d["text_raw"], hip))
        wers_txt.append(wer(d["text_raw"], hip))
        if len(ejemplos) < mostrar:
            ejemplos.append((_norm(d["text_raw"]), hip))

    for ref, hip in ejemplos:
        print(f"    [ctc] REF: {ref}\n    [ctc] HYP: {hip}")
    c = np.array(cers_txt)
    print(f"    por trial: mediana={np.median(c):.3f} p10={np.percentile(c,10):.3f} "
          f"p90={np.percentile(c,90):.3f} · {(c<0.2).mean()*100:.0f}% por debajo de 0.2")
    return err_bpe / max(1, ref_bpe), float(c.mean()), float(np.mean(wers_txt))


@torch.no_grad()
def evaluar(cfg, modelo, val_raw, n=EVAL_N, mostrar=3):
    """CER/WER con el beam search de OWSM (decoder + CTC)."""
    w_dec = cfg["ctc_weight_decode"]
    w_dec = cfg["ctc_weight"] if w_dec is None else w_dec
    s2t = get_s2t()
    rebind_beam_search(s2t, modelo, w_dec)

    lang_id   = s2t.converter.token2id[f"<{LANGUAGE}>"]
    task_id   = s2t.converter.token2id["<asr>"]
    notime_id = s2t.converter.token2id[s2t.preprocessor_conf["notime_symbol"]]

    nivel = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)   # el beam search es MUY verboso
    cers, wers, ejemplos = [], [], []
    try:
        n = min(n, len(val_raw))
        for i in range(n):
            d = val_raw[i]
            x = preparar_speech(d, cfg)
            speech = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            lens = torch.tensor([speech.shape[1]], dtype=torch.long, device=DEVICE)
            enc, _ = modelo.encode(speech, lens)
            if isinstance(enc, tuple):
                enc = enc[0]
            s2t.beam_search.set_hyp_primer([modelo.sos, lang_id, task_id, notime_id])
            res = s2t._decode_single_sample(enc[0])
            text, token, token_int, text_nospecial, hyp = res[0]
            hipotesis = (text_nospecial or text or "").strip()
            cers.append(cer(d["text_raw"], hipotesis))
            wers.append(wer(d["text_raw"], hipotesis))
            if i < mostrar:
                ejemplos.append((_norm(d["text_raw"]), hipotesis))
    finally:
        logging.getLogger().setLevel(nivel)

    for ref, hip in ejemplos:
        print(f"    REF: {ref}\n    HYP: {hip}")
    # Un modelo que ignora la senal sEEG produce siempre la misma frase generica:
    if ejemplos and len({h for _, h in ejemplos}) == 1:
        print("    [aviso] hipotesis identicas: el modelo esta ignorando la entrada")

    return float(np.mean(cers)), float(np.mean(wers)), n

print("Evaluacion lista.")

Evaluacion lista.


## 10. `run_experiment`

Une todo: datos → stats (cacheadas) → entrenamiento → métricas → guardado. Cada llamada deja en Drive una fila en `resultados.csv`, el `train.log` y un `epocas.csv` con la curva completa.

In [ ]:
CSV_RESULTADOS = f"{RESULTS_DIR}/resultados.csv"

def hacer_tag(cfg):
    extra = ""
    if cfg["frontend"] == "deep":
        extra += f"d{cfg['deep_dim']}b{cfg['deep_bloques']}s{cfg['subsample']}"
    if cfg["interctc_weight"] > 0:
        extra += f"_ic{cfg['interctc_weight']:g}"
    if cfg["aug"]:
        extra += "_aug"
        if cfg["aug_ruido"] > 0:
            extra += f"n{cfg['aug_ruido']:g}"
        if cfg.get("aug_offset", 0) > 0:
            extra += f"o{cfg['aug_offset']:g}"
    if cfg.get("suavizado", 0) > 0:
        extra += f"_sm{cfg['suavizado']:g}"
    if cfg["weight_decay"] >= 1e-4:
        extra += f"_wd{cfg['weight_decay']:g}"
    return (f"{cfg['nombre']}_n{cfg['n_sessions']}_{cfg['frontend']}{extra}"
            f"{'_sess' if cfg['capa_sesion'] else ''}"
            f"{'_char' if cfg['objetivo_ctc'] == 'char' else ''}"
            f"_unf{int(cfg['unfreeze_encoder'])}_lora{int(cfg['usar_lora'])}"
            f"_lr{cfg['lr']:g}_els{cfg['enc_lr_scale']:g}_ctc{cfg['ctc_weight']:g}"
            f"_bs{cfg['batch_size']}_ep{cfg['max_epoch']}")

def experimentos_hechos():
    if not os.path.exists(CSV_RESULTADOS):
        return set()
    try:
        return set(pd.read_csv(CSV_RESULTADOS)["exp_tag"].astype(str))
    except Exception:
        return set()

def guardar_resultado(fila):
    df_nueva = pd.DataFrame([fila])
    if os.path.exists(CSV_RESULTADOS):
        df = pd.concat([pd.read_csv(CSV_RESULTADOS), df_nueva], ignore_index=True)
    else:
        df = df_nueva
    df.to_csv(CSV_RESULTADOS, index=False)


def asegurar_stats(trainer, stats_dir):
    faltan = [p for p in shape_files(stats_dir)
              if not os.path.exists(p) or os.path.getsize(p) == 0]
    if not faltan:
        print("  shape files ya existen, se salta collect_stats")
        return
    print("  recopilando estadisticas (solo la primera vez por configuracion de datos)...")
    nivel = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)
    try:
        trainer.collect_stats()
    finally:
        logging.getLogger().setLevel(nivel)
    print("  collect_stats OK")


def run_experiment(cambios):
    cfg = {**DEFAULTS, **cambios}
    tag = hacer_tag(cfg)

    if SKIP_IF_DONE and tag in experimentos_hechos():
        print(f"[SALTADO] {tag}\n")
        return None

    print("=" * 78)
    print(f"[EXPERIMENTO] {tag}")
    print("=" * 78)

    exp_dir = f"{OUTPUT_DIR}/{tag}"
    os.makedirs(exp_dir, exist_ok=True)
    drive_dir = f"{RESULTS_DIR}/{tag}"
    os.makedirs(drive_dir, exist_ok=True)

    train_raw, val_raw, stats_dir, n_ses = get_datos(cfg)
    os.makedirs(stats_dir, exist_ok=True)

    data_info = make_data_info(cfg, aug=False)          # validacion: nunca aumentada
    train_ds = ez.dataset.ESPnetEZDataset(train_raw, data_info=make_data_info(cfg, aug=True))
    valid_ds = ez.dataset.ESPnetEZDataset(val_raw,   data_info=data_info)

    set_encoder_lr_scale(cfg["enc_lr_scale"])
    log_path = f"{exp_dir}/train.log"

    def entrenar(batch_size):
        """Un intento de entrenamiento con el batch dado."""
        c = {**cfg, "batch_size": batch_size}
        ft = build_finetune_config(c, stats_dir)
        pasos = max(1, len(train_raw) // max(1, batch_size))
        print(f"  batch={batch_size} · ~{pasos} pasos/epoca · warmup {c['warmup']} "
              f"(~{c['warmup']/pasos:.1f} epocas)")
        trainer = ez.Trainer(
            task="s2t", train_config=ft,
            train_dataset=train_ds, valid_dataset=valid_ds,
            build_model_fn=make_build_model_fn(c, n_ses), data_info=data_info,
            output_dir=exp_dir, stats_dir=stats_dir, ngpu=NGPU,
        )
        asegurar_stats(trainer, stats_dir)
        train_raw.cerrar(); val_raw.cerrar()   # antes del fork (HDF5 no es fork-safe)
        with CapturaLog(log_path):
            trainer.train()
        del trainer

    t0 = time.time()
    error, bs_usado = "", cfg["batch_size"]
    for intento, bs in enumerate([cfg["batch_size"], max(1, cfg["batch_size"] // 2),
                                  max(1, cfg["batch_size"] // 4)]):
        try:
            bs_usado = bs
            entrenar(bs)
            error = ""
            break
        except torch.cuda.OutOfMemoryError as e:
            gc.collect(); torch.cuda.empty_cache()
            error = f"OutOfMemoryError (batch={bs})"
            if intento < 2:
                print(f"  [OOM con batch={bs}] reintentando con batch={max(1, bs // 2)}...")
            else:
                print(f"  [ERROR] OOM incluso con batch={bs}. Usa una GPU con mas VRAM.")
        except KeyboardInterrupt:
            error = "interrumpido"
            print("  [interrumpido por el usuario]")
            break
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            print(f"  [ERROR] {error}")
            break
    minutos = (time.time() - t0) / 60

    # ── metricas por epoca ──
    df_ep = parse_train_log(log_path)
    mejores = {}
    if len(df_ep):
        df_ep.insert(0, "exp_tag", tag)
        df_ep.to_csv(f"{drive_dir}/epocas.csv", index=False)
        mejores["epocas_hechas"] = int(df_ep["epoch"].max())
        # El minimo de cer_ctc SOBRE TODAS LAS EPOCAS. Es lo que hay que mirar: el
        # cer_ctc "en la epoca de menor loss" puede ser mucho peor si la loss esta
        # dominada por la perdida de atencion.
        if "valid_cer_ctc" in df_ep.columns and df_ep["valid_cer_ctc"].notna().any():
            j = df_ep["valid_cer_ctc"].idxmin()
            mejores["min_valid_cer_ctc"] = float(df_ep.loc[j, "valid_cer_ctc"])
            mejores["epoch_min_cer_ctc"] = int(df_ep.loc[j, "epoch"])
            ult = df_ep.tail(10)
            if len(ult) >= 5:   # ¿sigue bajando o esta plano?
                mejores["pendiente_cer_ctc"] = float(
                    np.polyfit(ult["epoch"], ult["valid_cer_ctc"], 1)[0])
        if "valid_loss" in df_ep.columns and df_ep["valid_loss"].notna().any():
            mejor = df_ep.loc[df_ep["valid_loss"].idxmin()]
            for k in df_ep.columns:
                if k.startswith("valid_") and pd.notna(mejor[k]):
                    mejores[f"best_{k}"] = float(mejor[k])
            mejores["best_epoch"] = int(mejor["epoch"])

    # ── CER/WER: greedy CTC (honesto) y beam search (con decoder) ──
    cer_m = wer_m = cer_g = wer_g = cer_micro = float("nan")
    if not error or "interrumpido" in error:
        modelo = None
        try:
            modelo = cargar_modelo_entrenado(cfg, exp_dir, n_ses)

            if EVAL_GREEDY_CTC and modelo.ctc is not None:
                print("  evaluando con CTC greedy...")
                cer_micro, cer_g, wer_g = evaluar_ctc_greedy(cfg, modelo, val_raw)
                print(f"  [CTC greedy] cer_ctc(micro, comparable con el log)={cer_micro:.3f} · "
                      f"CER texto={cer_g:.3f} · WER texto={wer_g:.3f}")

            if getattr(modelo, "decoder", None) is not None and cfg["objetivo_ctc"] == "bpe":
                print("  evaluando con beam search...")
                cer_m, wer_m, n_ev = evaluar(cfg, modelo, val_raw)
                print(f"  [beam]       CER={cer_m:.3f} · WER={wer_m:.3f} ({n_ev} trials)")
            elif cfg["objetivo_ctc"] == "char":
                print("  (objetivo char: la cabeza CTC y el decoder usan vocabularios "
                      "distintos, el beam search hibrido no aplica)")
            else:
                print("  (sin decoder: ctc_weight=1, se omite el beam search)")
        except Exception as e:
            print(f"  [evaluacion fallida] {type(e).__name__}: {e}")
        finally:
            del modelo
            gc.collect(); torch.cuda.empty_cache()

    # ── guardado ──
    for artefacto in ["train.log", "config.yaml"]:
        origen = f"{exp_dir}/{artefacto}"
        if os.path.exists(origen):
            shutil.copy(origen, f"{drive_dir}/{artefacto}")
    ckpt_tmp = f"{exp_dir}/checkpoint.pth"          # modelo+optimizador, ~1.4 GB
    if GUARDAR_CKPT:
        try:
            mejor = find_best_checkpoint(exp_dir)
            # con su nombre real: si el criterio fue cer_ctc, llamarlo
            # valid.loss.best.pth confunde al releerlo meses despues
            shutil.copy(mejor, f"{drive_dir}/{os.path.basename(mejor)}")
            print(f"  checkpoint guardado en Drive: {os.path.basename(mejor)}")
        except Exception as e:
            print("  [aviso] no se pudo copiar el checkpoint:", e)
    if os.path.exists(ckpt_tmp) and not error:
        os.remove(ckpt_tmp)                          # libera disco de la VM

    fila = {"exp_tag": tag, "gpu": GPU_NAME, "minutos": round(minutos, 1),
            "compute_units": round(CU_H * minutos / 60, 2),
            "n_train": len(train_raw), "n_val": len(val_raw), "n_sesiones": n_ses,
            "batch_usado": bs_usado,
            "cer_ctc_micro": cer_micro,
            "cer_ctc_greedy": cer_g, "wer_ctc_greedy": wer_g,
            "cer_beam": cer_m, "wer_beam": wer_m, "error": error,
            **{k: v for k, v in cfg.items()}, **mejores}
    guardar_resultado(fila)

    del train_ds, valid_ds
    gc.collect(); torch.cuda.empty_cache()

    print(f"  hecho en {minutos:.1f} min (~{CU_H*minutos/60:.2f} compute units)\n")
    return fila

print("run_experiment listo.")

run_experiment listo.


## 11. El barrido

Cada entrada es solo lo que **cambia** respecto a `DEFAULTS`. El barrido está por fases: **lanza la fase 1 sola** y mira el resultado antes de seguir.

### Por qué el diagnóstico va primero

Lo que dijo tu primera tanda, en una línea: `acc` empieza en 0.557 y acaba en 0.578 tras 30 épocas. Esa `acc` es la del decoder **con teacher forcing** — se le dan los tokens correctos anteriores y se mide si acierta el siguiente. Un ~57 % es lo que saca OWSM usando solo estadística del inglés, sin mirar la entrada. Que arranque ahí y no se mueva significa que el encoder no aporta nada. Y `wer=1.000` con `cer=0.34` lo confirma desde el otro lado: escribe inglés bien formado y no acierta ni una palabra.

El `overfit` de la fase 1 elimina todas las salidas de escape a la vez: una sesión (sin deriva entre días), val = train (sin generalización que valga), `ctc_weight=1.0` (ESPnet desactiva el decoder, así que no hay prior lingüístico donde esconderse) y 200 épocas. Lo que quede es la capacidad del encoder de mapear sEEG → fonemas. Son ~10 min y 0.3 unidades.

### Por qué la capa por sesión

Es *la* característica de este dataset: 45 sesiones repartidas en 20 meses, y la señal que produce un mismo fonema cambia de un día a otro por deriva de los electrodos. El algoritmo de referencia de B2T'25 pasa las características por una capa lineal específica del día seguida de un `softsign` antes de nada más, y los trabajos cross-subject recientes usan transformadas afines por día para alinear todo a un espacio común.

Tú tenías un único frontend compartido para 10 sesiones: el modelo recibía diez mapeos señal→fonema contradictorios y tenía que deducir de qué día venía cada trial. `CapaSesion` aprende una matriz 512×512 más un sesgo por sesión, inicializados a la identidad, y aplica `softsign` después. El índice de sesión viaja hasta el frontend como un canal extra del tensor de entrada (512 → 513).

### Otros cambios respecto a la tanda anterior

- **`enc_lr_scale=0.01`** en los dos runs de la fase 2: el encoder preentrenado a 1e-5 y el frontend nuevo a 1e-3. Con todo a 1e-3 lo más probable es que el warmup estuviera destruyendo los pesos de OWSM.
- **`cer_ctc_greedy`** es la métrica a mirar en la tabla, no `cer_beam`. Sale de decodificar el `argmax` de la cabeza CTC colapsando repeticiones: sin decoder, sin beam, sin prior. Es la que no se puede falsear.

In [ ]:
BARRIDO = [
    # ══ v5 = v3 + suavizado temporal ════════════════════════════════
    # v3 llegó a 0.381 CER con aumento 0.5 + offset 0.2. El siguiente
    # paso natural es la palanca que dejamos sin estrenar: suavizado.
    #
    # En la literatura de B2T (Wilf, Anumanchipalli) el suavizado gaussiano
    # temporal es el preprocesado estándar, ~40 ms sigma. Aquí son bins de
    # 20 ms, así que suavizado=2.0 -> 40 ms. Se aplica igual en train y
    # validacion, NO es aumento, es preprocesado.
    #
    # Expectativa: pequeña mejora, 2-5% relativo. Sin riesgo. Si funciona
    # te deja en ~0.36 CER y con margen para v6 (subsample=4).
    #
    # COSTE: ~8 h, ~14 unidades. Sin cambios en batch_size ni memoria.
    dict(nombre="v5", n_sessions=45, max_epoch=60, warmup=1500,
         frontend="deep", deep_dim=512, deep_bloques=2, subsample=2,
         interctc_weight=0.3, interctc_layers=(2,),
         objetivo_ctc="char", ctc_weight=1.0, capa_sesion=True,
         norm="ninguna", usar_lora=False,
         batch_size=16, accum_grad=2,
         lr=1e-3, enc_lr_scale=0.05,
         optim="adamw", weight_decay=1e-2,
         grad_clip=100.0, patience=12, patience_start_epoch=15,
         aug=True, aug_ruido=0.5, aug_offset=0.2,
         aug_n_tiempo=2, aug_max_tiempo=0.05,
         aug_n_canales=2, aug_max_canales=20,
         suavizado=2.0),

    # ── v4 (ya hecho, resultó peor): CER 0.577 con aug_ruido=0.8.
    #    La relación señal/ruido con σ=0.8 sobre features std~1 cae a ~1.25,
    #    demasiado para aprender. El punto dulce es 0.5.
    #    No volver a tocar el ruido en este espacio.

    # ── v3 (ya hecho): cer_ctc 0.381 · CER greedy 0.325 · WER 0.632+LM 0.480
    #    v2 (ya hecho): cer_ctc 0.455 · CER greedy 0.396 · WER 0.675
]

# Calibrado con medidas reales en L4: v2 306 min/45 ep y v3 311 min/38 ep.
def estimar_min(cfg):
    c = {**DEFAULTS, **cfg}
    factor = 1.0 if "L4" in GPU_NAME else (2.0 if "T4" in GPU_NAME else 0.5)
    por_epoca = 2.28 * (c["n_sessions"] / 10) * (0.8 if c["ctc_weight"] == 1.0 else 1.0)
    return c["max_epoch"] * por_epoca * factor

total_min = sum(estimar_min(e) for e in BARRIDO)
print(f"GPU: {GPU_NAME} ({CU_H} unidades/h)")
print(f"{len(BARRIDO)} experimentos · ~{total_min/60:.1f} h · "
      f"~{CU_H*total_min/60:.1f} compute units (~${CU_H*total_min/60*0.10:.2f})\n")
for e in BARRIDO:
    print(f"  ~{estimar_min(e)/60:4.1f} h · {hacer_tag({**DEFAULTS, **e})}")

GPU: NVIDIA L4 (1.71 unidades/h)
1 experimentos · ~8.2 h · ~14.0 compute units (~$1.40)

  ~ 8.2 h · v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60


## 12. Ejecutar

Con Colab Pro+ puedes activar la ejecución en segundo plano y cerrar el navegador. Si te desconectan a mitad, vuelve a lanzar esta celda: los experimentos terminados se saltan y el que estuviera a medias reanuda desde su `checkpoint.pth`.

In [ ]:
inicio = time.time()
for cambios in BARRIDO:
    run_experiment(cambios)

print("=" * 78)
print(f"BARRIDO COMPLETO en {(time.time()-inicio)/60:.0f} min "
      f"(~{CU_H*(time.time()-inicio)/3600:.1f} compute units)")
print("Resultados en:", CSV_RESULTADOS)

[SALTADO] v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60

BARRIDO COMPLETO en 0 min (~0.0 compute units)
Resultados en: /content/drive/MyDrive/TFG/resultados/resultados.csv


## 13. Modelo de lenguaje sobre la salida CTC

Hasta aquí todas las métricas salen de **CTC greedy**: el carácter más probable en cada instante, sin nada detrás. Es la medida honesta de cuánto extrae el encoder de la señal, y por eso fue la correcta durante el diagnóstico. Pero produce texto como `you cat se to good at this pute is wee`, con palabras que no existen.

Esta sección añade la pieza que falta y que usa toda la literatura de brain-to-text: una **búsqueda en haz con un modelo de lenguaje de n-gramas**, que reordena las hipótesis del CTC favoreciendo secuencias de palabras reales.

Dos avisos importantes:

- **El n-grama se entrena SOLO con las transcripciones de entrenamiento.** Usar las de validación sería una fuga de datos que inflaría el resultado. El código lo construye desde `train_raw` y nunca toca `val_raw`.
- **Esto no mejora la lectura del cerebro, mejora la escritura.** En la memoria conviene reportar las dos cifras por separado: el CER greedy mide el encoder, el WER con LM mide el sistema completo. Mezclarlas oculta cuánto viene de cada parte.

`kenlm` instalado por pip solo trae el lector, no `lmplz`, así que el ARPA se genera aquí en Python puro (bigramas con backoff de Katz). Es autocontenido y reproducible.

In [ ]:
!pip install -q pyctcdecode https://github.com/kpu/kenlm/archive/master.zip

import math, collections, kenlm
from itertools import groupby
from pyctcdecode import build_ctcdecoder


def construir_arpa(frases, ruta, orden=2, k=0.4):
    """ARPA de orden 2 o 3 con backoff, en Python puro (el kenlm de pip no trae lmplz).

    Add-k en cada orden y pesos de backoff calculados para que la masa cuadre.
    No es Kneser-Ney; con 8.000 frases y ~3.700 palabras esa diferencia pesa
    menos que el salto de bigrama a trigrama, que es lo que interesa comparar.
    """
    N = orden
    cnt = {i: collections.Counter() for i in range(1, N + 1)}
    for f in frases:
        ws = ["<s>"] * (N - 1) + f.split() + ["</s>"]
        cnt[1].update(ws[N - 1:])                      # sin el relleno inicial
        for i in range(2, N + 1):
            cnt[i].update(tuple(ws[j:j + i]) for j in range(len(ws) - i + 1))

    vocab = sorted(cnt[1])
    V, tot = len(vocab) + 1, sum(cnt[1].values())      # +1 por <unk>
    p_unk = k / (tot + k * V)
    p = {1: {(w,): (cnt[1][w] + k) / (tot + k * V) for w in cnt[1]}}

    bo = {}
    for i in range(2, N + 1):
        ctx = collections.Counter()
        for g, c in cnt[i].items():
            ctx[g[:-1]] += c
        p[i] = {g: (c + k) / (ctx[g[:-1]] + k * V) for g, c in cnt[i].items()}
        num, den = collections.defaultdict(float), collections.defaultdict(float)
        for g in cnt[i]:
            num[g[:-1]] += p[i][g]
            den[g[:-1]] += p[i - 1].get(g[1:], p_unk)  # el mismo n-grama sin la cabeza
        for h in ctx:
            rb, ru = 1.0 - num[h], 1.0 - den[h]
            bo[h] = (rb / ru) if rb > 1e-9 and ru > 1e-9 else 1e-9

    lg = lambda x: math.log10(max(x, 1e-12))
    with open(ruta, "w") as f:
        f.write("\\data\\\n")
        f.write(f"ngram 1={len(vocab) + 2}\n")
        for i in range(2, N + 1):
            f.write(f"ngram {i}={len(p[i])}\n")

        f.write("\n\\1-grams:\n")
        f.write(f"{lg(p_unk):.6f}\t<unk>\n")
        f.write(f"-99\t<s>\t{lg(bo.get(('<s>',), 1.0)):.6f}\n")
        for w in vocab:
            b = bo.get((w,))
            f.write(f"{lg(p[1][(w,)]):.6f}\t{w}" + (f"\t{lg(b):.6f}" if b else "") + "\n")

        for i in range(2, N + 1):
            f.write(f"\n\\{i}-grams:\n")
            for g, pr in p[i].items():
                # convencion ARPA: un n-grama que predice <s> lleva -99
                col = "-99" if g[-1] == "<s>" else f"{lg(pr):.6f}"
                b = bo.get(g) if i < N else None
                f.write(f"{col}\t{' '.join(g)}" + (f"\t{lg(b):.6f}" if b else "") + "\n")
        f.write("\n\\end\\\n")

    # pyctcdecode quiere PALABRAS: <s> y </s> no lo son
    return ruta, [w for w in vocab if not w.startswith("<")]


def localizar_exp(tag):
    """El .pth puede estar en /content/exp (misma sesion) o en Drive (GUARDAR_CKPT)."""
    for base in (OUTPUT_DIR, RESULTS_DIR):
        d = os.path.join(base, tag)
        if os.path.isdir(d) and glob.glob(os.path.join(d, "*.pth")):
            return d
    raise FileNotFoundError(
        f"No encuentro ningun .pth de '{tag}' ni en {OUTPUT_DIR} ni en {RESULTS_DIR}. "
        f"Si la VM se reinicio y GUARDAR_CKPT estaba en False, el modelo se perdio.")


def _barra(i, n, etiqueta, cada=None):
    cada = cada or max(1, n // 10)
    if (i + 1) % cada == 0 or i + 1 == n:
        print(f"\r    {etiqueta}: {i + 1}/{n}", end="" if i + 1 < n else "\n", flush=True)


@torch.no_grad()
def logprobs_val(cfg, exp_dir, n=None):
    """Log-probabilidades CTC de la validacion, calculadas UNA sola vez.

    n=None -> los 1.426 trials. Ocupan poco (unos 35 KB por trial) y asi el
    barrido de alpha/beta y el oraculo no vuelven a tocar la GPU.
    """
    _, val_raw, _, n_ses = get_datos(cfg)
    modelo = cargar_modelo_entrenado(cfg, exp_dir, n_ses)
    n = len(val_raw) if n is None else min(n, len(val_raw))
    idxs = np.linspace(0, len(val_raw) - 1, n).astype(int)

    lps, refs = [], []
    for j, i in enumerate(idxs):
        d = val_raw[int(i)]
        x = preparar_speech(d, cfg)
        sp = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        ln = torch.tensor([sp.shape[1]], dtype=torch.long, device=DEVICE)
        enc, _ = modelo.encode(sp, ln)
        if isinstance(enc, tuple):
            enc = enc[0]
        # pyctcdecode espera log-probabilidades normalizadas en (T, V)
        lps.append(torch.log_softmax(modelo.ctc.ctc_lo(enc)[0].float(), -1).cpu().numpy())
        refs.append(d["text_raw"])
        _barra(j, n, "codificando")
    del modelo
    gc.collect(); torch.cuda.empty_cache()
    return lps, refs


def greedy_desde_lp(lp, blank=CHAR_BLANK_ID):
    """El MISMO greedy que evaluar_ctc_greedy: argmax + colapso + quitar blanks.

    Con pyctcdecode y beam_width=1 el resultado no es identico, y entonces las
    cifras no cuadrarian con la columna cer_ctc_greedy de resultados.csv.
    """
    ids = lp.argmax(-1).tolist()
    return destok_char([k for k, _ in groupby(ids) if k != blank])


def frases_entrenamiento(cfg):
    """Transcripciones de TRAIN normalizadas (nunca las de validacion: seria fuga)."""
    train_raw, _, _, _ = get_datos(cfg)
    return [_norm(t) for t in train_raw.textos()]


def wer_oraculo(dec_sin_lm, lps, refs, beam=100):
    """Cota inferior: el mejor WER alcanzable eligiendo dentro del beam acustico.

    Se usa el decodificador SIN modelo de lenguaje a proposito. Asi el beam
    contiene las hipotesis que propone el encoder por si solo, y el oraculo
    responde a: si tuviera un LM perfecto para reordenar esa lista, hasta donde
    llegaria. La diferencia entre oraculo y LM real es el margen que queda en el
    LM; la diferencia entre oraculo y cero es lo que le falta al encoder.
    """
    cer_o = wer_o = 0.0
    n_hip = 0
    for i, (lp, r) in enumerate(zip(lps, refs)):
        # pyctcdecode poda por defecto con token_min_logp=-5 y beam_prune_logp=-10.
        # Para un oraculo eso es demasiado estricto: interesa la lista mas amplia
        # que el encoder considere plausible, no la mas limpia.
        hips = [h[0] for h in dec_sin_lm.decode_beams(
            lp, beam_width=beam, token_min_logp=-8.0, beam_prune_logp=-20.0)]
        n_hip += len(hips)
        wers = [wer(r, h) for h in hips]
        j = int(np.argmin(wers))
        wer_o += wers[j]
        cer_o += cer(r, hips[j])
        _barra(i, len(lps), "oraculo")
    n = len(lps)
    return cer_o / n, wer_o / n, n_hip / n


def evaluar_con_lm(cfg, tag, ordenes=(2, 3), n=None, n_barrido=200,
                   alphas=(0.3, 0.6, 1.0), betas=(0.0, 1.5, 3.0),
                   beam=100, mostrar=3):
    """greedy vs. n-grama (bigrama y trigrama) vs. oraculo, sobre los mismos trials.

    alpha y beta se eligen sobre los primeros n_barrido trials y el numero final
    se da sobre los n trials completos. Ambos salen de validacion: hay que
    decirlo en la memoria, no hay test ciego todavia.
    """
    assert cfg["objetivo_ctc"] == "char", "el decodificador con LM asume objetivo char"
    exp_dir = localizar_exp(tag)
    print(f"Experimento: {exp_dir}\n")

    frases = frases_entrenamiento(cfg)
    lps, refs = logprobs_val(cfg, exp_dir, n=n)
    n = len(refs)
    labels = [""] + _CHARS                     # indice 0 = blank, igual que la cabeza CTC
    print(f"  {n} trials de validacion · LM sobre {len(frases)} frases de train\n")

    def medir(hips):
        return (float(np.mean([cer(r, h) for r, h in zip(refs, hips)])),
                float(np.mean([wer(r, h) for r, h in zip(refs, hips)])))

    filas = []
    cer_g, wer_g = medir([greedy_desde_lp(lp) for lp in lps])
    filas.append(("greedy", "", "", cer_g, wer_g))
    print(f"  greedy: CER {cer_g:.3f} · WER {wer_g:.3f}")

    hips_por_orden = {}
    for orden in ordenes:
        ruta, vocab = construir_arpa(frases, f"/content/lm{orden}.arpa", orden=orden)
        print(f"\n  {orden}-grama · {len(vocab)} palabras · "
              f"ARPA {os.path.getsize(ruta) / 1e6:.1f} MB")
        mejor = None
        for a in alphas:
            for b in betas:
                dec = build_ctcdecoder(labels, kenlm_model_path=ruta,
                                       unigrams=vocab, alpha=a, beta=b)
                sub = slice(0, min(n_barrido, n))
                hs = [dec.decode(lp, beam_width=beam) for lp in lps[sub]]
                w = float(np.mean([wer(r, h) for r, h in zip(refs[sub], hs)]))
                print(f"    alpha={a:<4} beta={b:<4} WER={w:.3f}  (n={min(n_barrido, n)})")
                if mejor is None or w < mejor[0]:
                    mejor = (w, a, b, dec)
        _, a, b, dec = mejor
        hips = []
        for i, lp in enumerate(lps):
            hips.append(dec.decode(lp, beam_width=beam))
            _barra(i, n, f"{orden}-grama final")
        c, w = medir(hips)
        hips_por_orden[orden] = hips
        filas.append((f"{orden}-grama", a, b, c, w))
        print(f"  {orden}-grama: CER {c:.3f} · WER {w:.3f}  (alpha={a}, beta={b})")

    dec_sin = build_ctcdecoder(labels)          # sin LM: el beam puramente acustico
    cer_o, wer_o, hip_medias = wer_oraculo(dec_sin, lps, refs, beam=beam)
    filas.append(("oraculo", "", "", cer_o, wer_o))
    print(f"  oraculo: CER {cer_o:.3f} · WER {wer_o:.3f}  "
          f"({hip_medias:.0f} hipotesis por trial de media)")

    print(f"\n  {'':12s} {'CER':>8} {'WER':>8}   ({n} trials)")
    for nom, a, b, c, w in filas:
        extra = f"   alpha={a}, beta={b}" if a != "" else ""
        print(f"  {nom:12s} {c:8.3f} {w:8.3f}{extra}")

    for k in range(mostrar):
        print(f"\n  REF    : {_norm(refs[k])}")
        print(f"  greedy : {greedy_desde_lp(lps[k])}")
        for orden in ordenes:
            print(f"  {orden}-grama: {hips_por_orden[orden][k]}")

    tabla = pd.DataFrame(filas, columns=["metodo", "alpha", "beta", "cer", "wer"])
    tabla.insert(0, "exp_tag", tag)
    tabla["n_trials"] = n
    destino = f"{RESULTS_DIR}/{tag}/lm_resultados.csv"
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    tabla.to_csv(destino, index=False)
    print(f"\n  guardado en {destino}")
    return tabla


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
espnet 202511 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


### Ejecucion

La celda de abajo hace las tres cosas: comprueba que el checkpoint existe, evalua
`greedy` contra `beam + n-grama` sobre los mismos 200 trials, y guarda la rejilla de
`(alpha, beta)` en Drive.

`alpha` pesa el modelo de lenguaje frente a la acustica y `beta` es un premio por
palabra insertada (compensa que el LM tiende a acortar). Los valores tipicos en
brain-to-text estan en `alpha` 0.3-1.0 y `beta` 0-3, que es justo lo que barre.
Elegirlos mirando el WER de validacion es lo estandar, pero **conviene decirlo en la
memoria**: son dos hiperparametros ajustados sobre validacion, no sobre un test ciego.

In [ ]:
# ══ EJECUTAR: greedy vs. bigrama vs. trigrama vs. oraculo ══════════════
# Requiere que la celda anterior (definiciones) se haya ejecutado.

cfg_eval = {**DEFAULTS, **BARRIDO[0]}      # la config con la que se entreno
TAG      = hacer_tag(cfg_eval)             # o pon la cadena del experimento a mano
print("Experimento:", TAG, "\n")

# ── 1. ¿sobrevive el checkpoint? ───────────────────────────────────────
hallados = {b: sorted(glob.glob(os.path.join(b, TAG, "*.pth")))
            for b in (OUTPUT_DIR, RESULTS_DIR)}
for base, ps in hallados.items():
    print(f"  {base}/{TAG}")
    print(f"    -> {[os.path.basename(p) for p in ps] if ps else 'sin checkpoints'}")

if not any(hallados.values()):
    print("\n" + "!" * 70)
    print("No hay checkpoint de este experimento: se entreno con GUARDAR_CKPT=False")
    print("y la VM se ha reiniciado desde entonces.")
    print("\nPara recuperarlo hay que reentrenar:")
    print("  1. GUARDAR_CKPT = True   (panel de control)  <- ya esta")
    print("  2. SKIP_IF_DONE = False  (si no, run_experiment lo salta)")
    print("  3. relanzar la celda 12 y volver aqui al terminar")
    print("!" * 70)
else:
    # ── 2. la evaluacion completa ──────────────────────────────────────
    # n=None -> los 1.426 trials de validacion, no una muestra de 200.
    # Solo el paso por la GPU es rapido; el resto es CPU y tarda un rato
    # (~15-25 min): dos ordenes de n-grama, nueve combinaciones de alpha/beta
    # cada uno sobre 200 trials, la pasada final sobre todos, y el oraculo.
    tabla = evaluar_con_lm(cfg_eval, TAG, ordenes=(2, 3), n=None, n_barrido=200)
    display(tabla)

Experimento: v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60 

  /content/exp/v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60
    -> ['60epoch.pth', 'latest.pth', 'valid.cer_ctc.ave.pth', 'valid.cer_ctc.ave_1best.pth', 'valid.cer_ctc.best.pth']
  /content/drive/MyDrive/TFG/resultados/v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60
    -> ['valid.cer_ctc.best.pth']
Experimento: /content/exp/v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60

Sesiones (45): ['t15.2023.08.11', 't15.2023.08.13', 't15.2023.08.18'] ...
  [aviso] no encontrado: /content/data/hdf5_data_final/t15.2023.08.11/data_val.hdf5
  [aviso] no encontrado: /content/data/hdf5_data_final/t15.2024.03.03/data_val.hdf5
  [aviso] no encontrado: /content/data/hdf5_data_final/t15.2024.04.25/data_val.hdf5
  [aviso] no encontr

INFO:root:Vocabulary size: 50002
INFO:root:Gradient checkpoint layers: []
INFO:root:Gradient checkpoint layers: []
/usr/local/lib/python3.12/dist-packages/espnet2/s2t/espnet_model.py:338: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):


    codificando: 1426/1426
  1426 trials de validacion · LM sobre 8072 frases de train

  greedy: CER 0.320 · WER 0.592

  2-grama · 3730 palabras · ARPA 0.5 MB
    alpha=0.3  beta=0.0  WER=0.440  (n=200)
    alpha=0.3  beta=1.5  WER=0.441  (n=200)
    alpha=0.3  beta=3.0  WER=0.442  (n=200)
    alpha=0.6  beta=0.0  WER=0.432  (n=200)
    alpha=0.6  beta=1.5  WER=0.434  (n=200)
    alpha=0.6  beta=3.0  WER=0.434  (n=200)
    alpha=1.0  beta=0.0  WER=0.435  (n=200)
    alpha=1.0  beta=1.5  WER=0.434  (n=200)
    alpha=1.0  beta=3.0  WER=0.440  (n=200)
    2-grama final: 1426/1426
  2-grama: CER 0.305 · WER 0.475  (alpha=0.6, beta=0.0)

  3-grama · 3730 palabras · ARPA 1.6 MB
    alpha=0.3  beta=0.0  WER=0.447  (n=200)
    alpha=0.3  beta=1.5  WER=0.447  (n=200)
    alpha=0.3  beta=3.0  WER=0.447  (n=200)
    alpha=0.6  beta=0.0  WER=0.443  (n=200)
    alpha=0.6  beta=1.5  WER=0.443  (n=200)
    alpha=0.6  beta=3.0  WER=0.443  (n=200)
    alpha=1.0  beta=0.0  WER=0.451  (n=200)
    alpha

,exp_tag,metodo,alpha,beta,cer,wer,n_trials
0,v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0....,greedy,,,0.320446,0.592450,1426
1,v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0....,2-grama,0.6,0.0,0.304971,0.475393,1426
2,v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0....,3-grama,0.6,1.5,0.308232,0.487583,1426
3,v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0....,oraculo,,,0.293327,0.503193,1426


## 14. Resultados y curvas

Tabla comparativa para la memoria y curvas de entrenamiento de cada experimento.

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_RESULTADOS)
cols = ["nombre", "frontend", "objetivo_ctc", "capa_sesion", "norm", "subsample",
        "interctc_weight", "grad_clip", "weight_decay", "lr", "enc_lr_scale",
        "ctc_weight", "n_train", "epocas_hechas",
        "min_valid_cer_ctc", "epoch_min_cer_ctc", "pendiente_cer_ctc",
        "cer_ctc_micro", "cer_ctc_greedy", "wer_ctc_greedy", "cer_beam", "wer_beam",
        "minutos", "compute_units"]
cols = [c for c in cols if c in df.columns]
# ordenar por lo que de verdad se compara: el mejor cer_ctc de validacion
clave = ("min_valid_cer_ctc" if "min_valid_cer_ctc" in df.columns
         else ("cer_ctc_greedy" if "cer_ctc_greedy" in df.columns else "nombre"))
resumen = df[cols].sort_values(clave)
display(resumen)

resumen.to_csv(f"{RESULTS_DIR}/tabla_resumen.csv", index=False)
print("Tabla guardada en", f"{RESULTS_DIR}/tabla_resumen.csv")
print("\nLaTeX para la memoria:\n")
print(resumen.to_latex(index=False, float_format="%.3f"))

,nombre,frontend,objetivo_ctc,capa_sesion,norm,subsample,interctc_weight,grad_clip,weight_decay,lr,...,min_valid_cer_ctc,epoch_min_cer_ctc,pendiente_cer_ctc,cer_ctc_micro,cer_ctc_greedy,wer_ctc_greedy,cer_beam,wer_beam,minutos,compute_units
12,charOverfit,linear,char,False,trial,NaN,NaN,5.0,0.000001,0.001,...,0.052,75.0,-0.003467,0.461976,0.450994,0.779190,NaN,NaN,25.7,0.73
14,charOverfit,linear,char,False,trial,NaN,NaN,5.0,0.000001,0.001,...,0.231,50.0,-0.018545,0.538735,0.537149,0.886817,NaN,NaN,17.3,0.49
24,v5,deep,char,True,ninguna,2.0,0.3,100.0,0.010000,0.001,...,0.362,60.0,-0.001212,0.318539,0.313652,0.584330,NaN,NaN,531.6,15.15
22,v3,deep,char,True,ninguna,2.0,0.3,100.0,0.010000,0.001,...,0.381,37.0,-0.003527,0.332871,0.324703,0.631565,NaN,NaN,310.7,8.86
21,v2,deep,char,True,ninguna,2.0,0.3,25.0,0.010000,0.001,...,0.455,43.0,-0.000933,0.401603,0.395745,0.674820,NaN,NaN,306.2,8.73
20,deep,deep,char,True,trial,4.0,0.3,5.0,0.000001,0.001,...,0.514,86.0,0.000200,0.437384,0.431454,0.715029,NaN,NaN,584.8,16.67
23,v4,deep,char,True,ninguna,2.0,0.3,100.0,0.010000,0.001,...,0.577,41.0,-0.004830,0.504546,0.501666,0.790443,NaN,NaN,340.6,9.71
11,gen1,linear,NaN,False,trial,NaN,NaN,5.0,0.000001,0.001,...,0.753,200.0,-0.000903,0.768543,0.795170,0.969841,NaN,NaN,51.0,1.45
15,charGen1,linear,char,False,trial,NaN,NaN,5.0,0.000001,0.001,...,0.784,68.0,-0.000164,0.725092,0.734234,0.972857,NaN,NaN,34.8,0.99
18,c1d,conv1d,char,True,trial,NaN,NaN,5.0,0.000001,0.001,...,0.804,29.0,-0.004000,0.713757,0.714165,1.204857,NaN,NaN,168.9,4.81


Tabla guardada en /content/drive/MyDrive/TFG/resultados/tabla_resumen.csv

LaTeX para la memoria:

\begin{tabular}{lllllrrrrrrrrrrrrrrrrrrr}
\toprule
nombre & frontend & objetivo_ctc & capa_sesion & norm & subsample & interctc_weight & grad_clip & weight_decay & lr & enc_lr_scale & ctc_weight & n_train & epocas_hechas & min_valid_cer_ctc & epoch_min_cer_ctc & pendiente_cer_ctc & cer_ctc_micro & cer_ctc_greedy & wer_ctc_greedy & cer_beam & wer_beam & minutos & compute_units \\
\midrule
charOverfit & linear & char & False & trial & NaN & NaN & 5.000 & 0.000 & 0.001 & 1.000 & 1.000 & 348 & 75.000 & 0.052 & 75.000 & -0.003 & 0.462 & 0.451 & 0.779 & NaN & NaN & 25.700 & 0.730 \\
charOverfit & linear & char & False & trial & NaN & NaN & 5.000 & 0.000 & 0.001 & 1.000 & 1.000 & 348 & 50.000 & 0.231 & 50.000 & -0.019 & 0.539 & 0.537 & 0.887 & NaN & NaN & 17.300 & 0.490 \\
v5 & deep & char & True & ninguna & 2.000 & 0.300 & 100.000 & 0.010 & 0.001 & 0.050 & 1.000 & 8072 & 60.000 & 0.362 & 60.000

In [ ]:
# Curvas de todos los experimentos con epocas.csv
# OJO: con ctc_weight=1.0 ESPnet desactiva el decoder de atencion, asi que en el
# log NO existe valid_cer (esa es la del decoder): la metrica es valid_cer_ctc.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for tag in df["exp_tag"]:
    p = f"{RESULTS_DIR}/{tag}/epocas.csv"
    if not os.path.exists(p):
        continue
    e = pd.read_csv(p)
    etiqueta = tag.split("_")[0]
    col_tr = "train_loss_ctc" if "train_loss_ctc" in e else "train_loss"
    col_va = "valid_loss_ctc" if "valid_loss_ctc" in e else "valid_loss"
    col_er = "valid_cer_ctc"  if "valid_cer_ctc"  in e else "valid_cer"
    if col_tr in e: axes[0].plot(e["epoch"], e[col_tr], label=etiqueta)
    if col_va in e: axes[1].plot(e["epoch"], e[col_va], label=etiqueta)
    if col_er in e: axes[2].plot(e["epoch"], e[col_er], label=etiqueta)

for ax, t in zip(axes, ["Loss CTC (train)", "Loss CTC (valid)", "CER CTC (valid)"]):
    ax.set_title(t); ax.set_xlabel("epoca"); ax.grid(alpha=0.3)
axes[2].set_ylim(0, 1.05)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/curvas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en", f"{RESULTS_DIR}/curvas.png")

Figura guardada en /content/drive/MyDrive/TFG/resultados/curvas.png


---

## Notas de uso

### Orden de ejecucion

Las celdas dependen unas de otras: la seccion 5 usa los imports de la 3 y las clases
de la 4. Si ejecutas "celda y siguientes" desde la mitad del notebook, revienta con un
`NameError`. Lo normal es **Entorno de ejecucion -> Ejecutar todas** despues del
reinicio que pide la celda de instalacion.

### Que mirar en `resultados.csv`

- **`min_valid_cer_ctc`**: el minimo de `cer_ctc` sobre TODAS las epocas. Es la cifra
  de referencia para comparar experimentos entre si.
- **`cer_ctc_greedy` / `wer_ctc_greedy`**: decodificacion greedy sobre el mejor
  checkpoint, sin decoder ni beam ni prior linguistico. Es la que no se puede falsear
  y la que mide lo que el encoder extrae del sEEG.
- **`pendiente_cer_ctc`**: ajuste lineal sobre las ultimas 10 epocas. Distingue "va
  lento" (negativa) de "esta atascado" (~0). Si sigue bajando al acabar, sube
  `max_epoch` antes de tocar nada mas.
- El WER con LM (seccion 13) va aparte: mide el sistema completo, no el encoder.
  En la memoria conviene reportar las dos cifras por separado.

### Checkpoints

Con `GUARDAR_CKPT = True` el mejor `.pth` (~470 MB) se copia a Drive al terminar cada
experimento. Con `False` vive solo en `/content` y desaparece con la VM: te quedarias
sin poder ejecutar la seccion 13 ni sacar ejemplos de hipotesis para la memoria.

### Ajustes de rendimiento

- Si la GPU va a menos del 70 % (`!nvidia-smi`), sube `batch_size` o pasa a
  `batch_type="numel"` con `batch_bins` (agrupa por numero de elementos, mucho mejor
  con longitudes de 138 a 1966 frames).
- `num_workers=0` es lo seguro con HDF5. Subelo solo si ves la GPU esperando.
- El reintento automatico por OOM baja el batch a la mitad y luego a un cuarto antes
  de rendirse, asi que un experimento mal dimensionado no tumba el barrido entero.
- `hacer_tag` no incluye `norm`, `optim` ni `accum_grad`: si repites un experimento
  cambiando SOLO uno de esos, el tag coincide y `SKIP_IF_DONE` lo saltara. Cambia
  tambien `nombre` en esos casos.